# Image Editing LLM Pipeline — Phase 4: Benchmark Evaluation (v1.4)

**Purpose:** Score the Phase 3 inference pipeline against the **singleturn** split of the ImgEdit benchmark.

**v1.4 changes (adjust-filter):**
- `CFG.bench_edit_type_filter` added to §0.1 — set to `('adjust',)` by default.
- §3.3 applies the filter after loading/caching `bench_samples` so only `adjust` samples are evaluated.
- Cache is automatically invalidated when the filter changes.
- §8 report header and `bench_report.json` version field updated to `1.4`.

**v1.3 baseline (singleturn-only):**
- §3.3 parser loads `Benchmark/singleturn/singleturn.json` directly — all 737 images

## §0 — Global Configuration
*Extends Phase 3 CFG. Only edit this cell.*

In [ ]:
# ── §0.1  GLOBAL CONFIG — edit this cell only ──────────────────────────────
# Phase 4 additions are marked with: # ← Phase 4
# v1.1 additions are marked with:    # ← v1.1

import os, sys, json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class PipelineConfig:
    # ── Drive root ────────────────────────────────────────────────────────
    drive_root: Path = Path('/content/drive/MyDrive/img_edit_pipeline')

    # ── Dataset (training split, kept for shared CFG structure) ───────────
    hf_dataset_id: str         = 'sysuyy/ImgEdit'
    hf_configs: Optional[list] = None
    n_subset: int              = 10_000

    expected_edit_types: tuple = (
        'action', 'add', 'adjust', 'background', 'content',
        'hybrid', 'reference', 'remove', 'replace', 'style', 'version',
    )
    mask_dataset_id: str = 'sysuyy/ImgEdit_recap_mask'

    global_adjust_keywords: tuple = (
        'lighting', 'light', 'brightness', 'exposure', 'contrast',
        'saturation', 'hue', 'tone', 'tones', 'overall', 'scene',
        'atmosphere', 'color temperature', 'white balance',
    )

    # ── Phase 1 model ─────────────────────────────────────────────────────
    vlm_model_id: str       = 'Qwen/Qwen2.5-VL-3B-Instruct'
    vlm_quant_bits: int     = 4
    vlm_lora_r: int         = 16
    vlm_lora_alpha: int     = 32
    vlm_lora_dropout: float = 0.05
    vlm_hidden_dim: int     = 2048

    # ── Phase 2 model ─────────────────────────────────────────────────────
    sd_model_id: str        = 'runwayml/stable-diffusion-inpainting'
    sd_cross_attn_dim: int  = 768
    sd_lora_r: int          = 8
    sd_lora_alpha: int      = 16
    sd_lora_dropout: float  = 0.0
    sd_lora_target_modules: tuple = ('to_q', 'to_k', 'to_v', 'to_out.0')
    vae_scale_factor: float = 0.18215
    phase2_resolution: int  = 512

    # ── Phase 3 inference ─────────────────────────────────────────────────
    phase3_resolution: int          = 512
    phase3_num_inference_steps: int = 20
    phase3_guidance_scale: float    = 7.5
    phase3_max_new_tokens: int      = 256
    phase3_sam2_model_cfg: str      = 'sam2.1/sam2.1_hiera_l'
    phase3_mixed_precision: str     = 'fp16'
    phase3_fallback_bbox: tuple     = (0, 0, 1000, 1000)

    grounding_model_id: str         = 'IDEA-Research/grounding-dino-tiny'
    grounding_box_threshold: float  = 0.25
    grounding_text_threshold: float = 0.20
    grounding_max_boxes: int        = 5
    bbox_degenerate_max_frac: float = 0.7
    bbox_degenerate_min_frac: float = 0.001

    sam2_multimask_output: bool     = True
    sam2_use_centre_point: bool     = True
    sam2_clip_rerank: bool          = True
    mask_morph_close_kernel: int    = 7
    mask_min_component_area_frac: float = 0.002
    mask_keep_largest_component: bool   = True
    global_edit_area_frac: float    = 0.90

    # ── Phase 4 evaluation ────────────────────────────────────────────────
    # Filename of the benchmark tar in the HF repo root.
    bench_tar_filename: str    = 'Benchmark.tar'  # ← Phase 4

    # Which top-level split to evaluate.                        ← v1.1
    # 'singleturn' | 'hard'
    # 'multiturn' is NOT supported (requires multi-step dialogue).
    bench_split: str           = 'singleturn'      # ← v1.1

    # How many samples to evaluate. Start at 20 for a smoke run.
    # Set to None to run ALL samples in the split (737 for singleturn).
    bench_n_samples: int       = 500               # ← Phase 4

    # Filter to a subset of edit types before inference.             # ← v1.4
    # Set to None to run ALL edit types (default v1.3 behaviour).
    # Example: ('adjust',) evaluates only colour/tone/lighting edits.
    bench_edit_type_filter: Optional[tuple] = ('adjust',)  # ← v1.4

    # Edit types present in the singleturn split.              ← v1.1
    # These match the keys in Benchmark/singleturn/singleturn.json.
    singleturn_edit_types: tuple = (               # ← v1.1
        'action', 'add', 'adjust', 'background',
        'compose', 'extract', 'remove', 'replace', 'style',
    )

    # Use edit-type-specific judge rubrics from judge_prompt.json.  ← v1.1
    # When True, the judge receives the rubric designed for each edit_type.
    # When False, the generic four-criterion prompt is used.
    bench_use_type_specific_judge: bool = True     # ← v1.1

    # CLIP model for CLIP-I similarity metric.
    clip_judge_model_id: str   = 'openai/clip-vit-large-patch14'  # ← Phase 4

    # LLM judge: False = use base Qwen2.5-VL (less biased; recommended).
    judge_use_lora: bool       = False            # ← Phase 4
    judge_max_new_tokens: int  = 512              # ← Phase 4
    judge_score_min: int       = 1                # ← Phase 4
    judge_score_max: int       = 5                # ← Phase 4

    # ── Training (kept for shared CFG structure) ──────────────────────────
    phase1_epochs: int         = 3
    phase1_batch_size: int     = 2
    phase1_grad_accum: int     = 8
    phase1_lr: float           = 2e-4
    phase1_max_seq_len: int    = 2560
    phase1_max_img_pixels: int = 802_816
    phase1_warmup_steps: int   = 100
    phase1_eval_steps: int     = 200
    phase2_epochs: int         = 5
    phase2_batch_size: int     = 1
    phase2_grad_accum: int     = 16
    phase2_lr: float           = 1e-4
    phase2_warmup_steps: int   = 200
    phase2_shard_size: int     = 5_000
    phase2_max_grad_norm: float = 1.0
    phase2_num_workers: int    = 2
    phase2_save_steps: int     = 500
    phase2_mixed_precision: str = 'fp16'
    phase2_num_train_timesteps: int = 1000
    phase2_beta_schedule: str  = 'linear'
    max_invalid_rle_frac: float = 0.05
    max_fallback_frac: float    = 0.15

    # ── Derived paths ─────────────────────────────────────────────────────
    @property
    def data_dir(self) -> Path:
        return self.drive_root / 'data' / 'imgedit_subset'

    @property
    def benchmark_dir(self) -> Path:
        return self.drive_root / 'data' / 'benchmark'

    @property
    def benchmark_extracted_dir(self) -> Path:
        return self.benchmark_dir / 'extracted'

    @property
    def benchmark_split_dir(self) -> Path:
        """Root directory for the chosen split inside the extracted tar."""
        return self.benchmark_extracted_dir / 'Benchmark' / self.bench_split

    @property
    def ckpt_phase1(self) -> Path:
        return self.drive_root / 'checkpoints' / 'phase1_vlm'

    @property
    def ckpt_phase2(self) -> Path:
        return self.drive_root / 'checkpoints' / 'phase2_diffusion_v2_r16'

    @property
    def ckpt_phase2_final(self) -> Path:
        return self.ckpt_phase2 / 'final'

    @property
    def outputs_eval(self) -> Path:
        return self.drive_root / 'outputs' / 'eval_v3'

    @property
    def outputs_bench(self) -> Path:
        # v1.4: directory encodes split + active filter for clarity.
        if self.bench_edit_type_filter:
            _tag = '_'.join(sorted(self.bench_edit_type_filter))
            return self.drive_root / 'outputs' / f'bench_v4_{self.bench_split}_{_tag}'
        return self.drive_root / 'outputs' / f'bench_v4_{self.bench_split}'

    @property
    def outputs_scores(self) -> Path:
        return self.drive_root / 'outputs' / 'scores'

    @property
    def manifest_path(self) -> Path:
        return self.data_dir / 'samples.json'

    @property
    def filtered_manifest_path(self) -> Path:
        return self.data_dir / 'samples_filtered.json'

    @property
    def hidden_states_dir(self) -> Path:
        return self.data_dir / 'vlm_hidden_states'

    @property
    def audit_path(self) -> Path:
        return self.data_dir / 'audit_report.json'

    @property
    def sam2_dir(self) -> Path:
        return self.drive_root / 'sam2'

    @property
    def sam2_checkpoint(self) -> Path:
        return self.sam2_dir / 'sam2.1_hiera_large.pt'

    @property
    def bench_results_path(self) -> Path:
        return self.outputs_bench / 'bench_inference_results.json'

    @property
    def bench_scores_path(self) -> Path:
        return self.outputs_bench / 'bench_scores.json'


CFG = PipelineConfig()
print('Config loaded (Phase 4 v1.4 — singleturn split, adjust filter).')
print(f'  Drive root           : {CFG.drive_root}')
print(f'  Benchmark dir        : {CFG.benchmark_dir}')
print(f'  Benchmark extracted  : {CFG.benchmark_extracted_dir}')
print(f'  Benchmark split dir  : {CFG.benchmark_split_dir}')
print(f'  Bench split          : {CFG.bench_split}')
print(f'  Bench outputs        : {CFG.outputs_bench}')
print(f'  Bench N samples      : {CFG.bench_n_samples}  (None = all 737)')
print(f'  Edit type filter     : {CFG.bench_edit_type_filter}  (None = all types)')
print(f'  Type-specific judge  : {CFG.bench_use_type_specific_judge}')
print(f'  Benchmark tar file   : {CFG.bench_tar_filename} (50.3 MB on HF)')
print(f'  Phase 1 ckpt         : {CFG.ckpt_phase1}')
print(f'  Phase 2 ckpt         : {CFG.ckpt_phase2_final}')
print(f'  CLIP judge model     : {CFG.clip_judge_model_id}')
print(f'  Judge uses LoRA      : {CFG.judge_use_lora}')


## §0.2 — Google Drive Mount & Directory Scaffold

In [ ]:
# ── §0.2  Mount Drive and create Phase 4 directories ────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

_dirs = [
    CFG.data_dir,
    CFG.benchmark_dir,
    CFG.outputs_bench,
    CFG.outputs_scores,
    CFG.ckpt_phase1,
    CFG.ckpt_phase2_final,
    CFG.sam2_dir,
]
for d in _dirs:
    d.mkdir(parents=True, exist_ok=True)

print('Drive mounted. Directories verified:')
for d in _dirs:
    print(f'  {"OK" if d.exists() else "MISSING"} {d}')


## §1 — Package Installation
⚠ **Run once per runtime.** Restart kernel after this cell, then continue from §1.2.

Installs all Phase 3 and Phase 4 dependencies in one step (self-contained — no need to run Phase 3 notebook first).

In [ ]:
# ── §1.1  Install pinned dependencies (Phase 3 + Phase 4) ───────────────
# After this cell finishes: RESTART KERNEL, then run §1.2.

import subprocess, sys

def _install(pkg, label=None):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    tag = label or pkg
    if r.returncode == 0:
        print(f"  ok  {tag}")
    else:
        print(f"  FAILED  {tag}")
        print(r.stderr[-400:])

# ── Step 1: numpy<2 FIRST ─────────────────────────────────────────────────
print("Step 1: numpy<2 (must come before all other installs)...")
_install("numpy<2", "numpy<2")

# ── Step 2: Core packages ─────────────────────────────────────────────────
print("\nStep 2: Phase 3 packages...")
for pkg, label in [
    ("diffusers>=0.27.0",     "diffusers>=0.27"),
    ("peft>=0.10.0",          "peft>=0.10"),
    ("accelerate>=0.28.0",    "accelerate>=0.28"),
    ("transformers>=4.49.0",  "transformers>=4.49"),
    ("bitsandbytes>=0.44.0",  "bitsandbytes>=0.44"),
    ("pycocotools>=2.0.7",    "pycocotools>=2.0.7"),
    ("tqdm>=4.66",            "tqdm>=4.66"),
    ("Pillow>=10.0",          "Pillow>=10"),
    ("qwen-vl-utils>=0.0.8",  "qwen-vl-utils>=0.0.8"),
]:
    _install(pkg, label)

# ── Step 3: torchao fix (PEFT LoRA on Colab) ─────────────────────────────
print("\nStep 3: torchao>=0.16.0 (PEFT LoRA compatibility fix)...")
_install("torchao>=0.16.0", "torchao>=0.16.0")

# ── Step 4: SAM2 from GitHub ─────────────────────────────────────────────
# SAM2 is not on PyPI — must be installed from source.
# We clone to /content/sam2 (Colab local, lost on restart).
# The Drive-cached checkpoint (CFG.sam2_checkpoint) persists across restarts.
#
# SAM2 install fragility notes:
#   - torch>=2.5.1 required (spec §1.2 REQUIRED dict confirms this)
#   - The `sam2` package installs Hydra for config resolution
#   - After kernel restart, re-run only §21.1 (SAM2 re-install) — checkpoint
#     is loaded from Drive so no re-download needed.
print("\nStep 4: SAM2 from GitHub...")
_sam2_dir = "/content/sam2"
import os
if not os.path.exists(_sam2_dir):
    r = subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/facebookresearch/sam2.git", _sam2_dir],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"  FAILED: git clone sam2: {r.stderr[-300:]}")
    else:
        print("  ok  git clone sam2")
else:
    print("  ok  /content/sam2 already exists")

_r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", _sam2_dir],
    capture_output=True, text=True
)
if _r.returncode == 0:
    print("  ok  sam2 installed from source")
else:
    print(f"  FAILED: pip install sam2: {_r.stderr[-300:]}")

# ── Phase 4 additions ──────────────────────────────────────────────────
print("\nStep 5: Phase 4 evaluation packages (lpips, scikit-image, torchmetrics)...")
_install('lpips',                 'lpips (perceptual metric)')
_install('scikit-image>=0.21.0',  'scikit-image (SSIM)')
_install('torchmetrics',          'torchmetrics')

print()
print("Installation complete.")
print("⚠  RESTART KERNEL NOW, then run §1.2.")
print("   Runtime → Restart session  (Ctrl+M .)  then continue from §1.2.")


## §1.2 — Import & Version Sanity Checks
*Run after every kernel restart. All checks must pass.*

In [ ]:
# ── §1.2  Import and version gate ───────────────────────────────────────────
import importlib, importlib.metadata, sys
from pathlib import Path

_failures = []

def _ver(pkg_name):
    try:
        return importlib.metadata.version(pkg_name)
    except importlib.metadata.PackageNotFoundError:
        return None

def _require(pkg_name, min_ver=None):
    v = _ver(pkg_name)
    if v is None:
        _failures.append(f'MISSING  {pkg_name}')
        return
    if min_ver:
        from packaging.version import Version
        if Version(v) < Version(min_ver):
            _failures.append(f'TOO OLD  {pkg_name}=={v} (need >={min_ver})')
            return
    print(f'  OK  {pkg_name}=={v}')

print('Checking required packages...')
_require('torch',          '2.0.0')
_require('transformers',   '4.40.0')
_require('diffusers',      '0.27.0')
_require('peft',           '0.10.0')
_require('datasets',       '2.14.0')
_require('Pillow',         '9.0.0')
_require('numpy',          '1.24.0')
_require('lpips',          '0.1.4')
_require('scikit-image',   '0.21.0')

if _failures:
    print('\n=== FAILURES ===')
    for f in _failures:
        print(f'  {f}')
    raise RuntimeError(
        f'{len(_failures)} package(s) missing or too old. '
        'Re-run §1.1 and restart kernel.'
    )
print('All package checks passed.')

import io, time, logging, traceback
import numpy as np
import torch
from PIL import Image
from typing import Optional, Dict, List, Tuple

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger('phase4')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('  WARNING: No GPU detected. Inference will be very slow.')


## §2 — Phase 3 Core Utilities & Model Loading
*Embedded from Phase 3 v3.0. These cells are byte-identical to Phase 3 and must not be modified.*

## §3.1 — Core Utilities (Phase 3 Subset)
Includes RLE decode, bbox helpers, `QWEN_SPECIAL_START`, `load_json`, `save_json`.

Warning: `QWEN_SPECIAL_START = 151643` must match Phase 1 §11.1 — do not change.

In [ ]:
# -- §3.1  Core utilities -- Phase 3 subset ----------------------------------------
# CRITICAL: bbox helpers and QWEN_SPECIAL_START must be byte-identical
# to Phase 1 §3.1 and §11.1. Any change breaks train/inference consistency.

import io, json, logging
import numpy as np
from pathlib import Path
from typing import Any, Optional
from PIL import Image

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger('pipeline')

import pycocotools.mask as mask_utils

# Token IDs < 151643 : regular text tokens -- KEPT for mean pooling
# Token IDs >= 151643: special tokens (<|image_pad|>, <|im_start|>, etc.) -- EXCLUDED
# Confirmed from Phase 1 §11.1. Do NOT change without recomputing all hidden states.
QWEN_SPECIAL_START = 151643

KNOWN_EDIT_TYPES       = set(CFG.expected_edit_types)
GLOBAL_ADJUST_KEYWORDS = set(CFG.global_adjust_keywords)


def decode_rle_mask(rle: dict) -> Optional[np.ndarray]:
    '''Decode COCO-format RLE dict to binary uint8 mask (H x W). Returns None on failure.'''
    if rle is None:
        return None
    try:
        rle_copy = dict(rle)
        if isinstance(rle_copy.get('counts'), str):
            rle_copy['counts'] = rle_copy['counts'].encode('utf-8')
        mask = mask_utils.decode(rle_copy)
        assert mask.ndim == 2, f'Expected 2-D mask, got {mask.shape}'
        return mask
    except Exception as e:
        log.debug(f'RLE decode failed: {e}')
        return None


def bbox_area_absolute(bbox_xyxy: list) -> float:
    x1, y1, x2, y2 = bbox_xyxy
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def clip_bbox_to_image(bbox_xyxy: list, W: int, H: int) -> list:
    x1, y1, x2, y2 = bbox_xyxy
    return [max(0, min(x1, W)), max(0, min(y1, H)),
            max(0, min(x2, W)), max(0, min(y2, H))]


def bbox_to_relative(bbox_xyxy: list, W: int, H: int) -> list:
    x1, y1, x2, y2 = clip_bbox_to_image(bbox_xyxy, W, H)
    return [round(x1/W*1000), round(y1/H*1000), round(x2/W*1000), round(y2/H*1000)]


def bbox_relative_to_absolute(bbox_rel: list, W: int, H: int) -> list:
    '''Convert [0,1000] relative bbox to absolute pixel coords.

    Used in Phase 3 to convert VLM bbox output to SAM2 input.
    SAM2 requires absolute pixel coordinates; VLM outputs [0,1000] relative scale.
    Clamps to image bounds to handle out-of-range VLM predictions.
    '''
    x1, y1, x2, y2 = bbox_rel
    abs_bbox = [round(x1/1000*W), round(y1/1000*H), round(x2/1000*W), round(y2/1000*H)]
    return clip_bbox_to_image(abs_bbox, W, H)


def is_global_edit(edit_type: str, edit_description: str) -> bool:
    '''Return True if this is a global (full-image) edit -- no localised mask needed.'''
    et = (edit_type or '').strip().lower()
    if et in {'style', 'background'}:
        return True
    if et == 'adjust':
        desc_lower = (edit_description or '').lower()
        if any(kw in desc_lower for kw in GLOBAL_ADJUST_KEYWORDS):
            return True
    return False


def is_full_image_bbox(bbox_rel: list) -> bool:
    '''Return True if bbox covers >=95% of the full image in [0,1000] scale.
    Used to detect when VLM predicted a full-image bbox (global edit).
    '''
    x1, y1, x2, y2 = bbox_rel
    return x1 <= 50 and y1 <= 50 and x2 >= 950 and y2 >= 950


def save_json(obj: Any, path: Path, indent: int = 2) -> None:
    '''Atomic JSON write (tmp then rename).'''
    path = Path(path)
    tmp  = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, indent=indent, ensure_ascii=False))
    tmp.rename(path)


def load_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Expected JSON not found: {path}')
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError as e:
        raise ValueError(f'Corrupt JSON at {path}: {e}') from e


# -- Smoke test -----------------------------------------------------------
assert QWEN_SPECIAL_START == 151643
assert bbox_relative_to_absolute([500, 250, 750, 875], 1920, 1080) == [960, 270, 1440, 945]
_rel = bbox_to_relative([960, 270, 1440, 945], 1920, 1080)
assert _rel == [500, 250, 750, 875], f'Round-trip bbox failed: {_rel}'
assert is_global_edit('style', 'make it painterly')
assert not is_global_edit('remove', 'remove the car')
assert is_full_image_bbox([0, 0, 1000, 1000])
assert not is_full_image_bbox([100, 100, 800, 800])
print('§3.1 utilities loaded:')
print(f'  QWEN_SPECIAL_START = {QWEN_SPECIAL_START}')
print('  decode_rle_mask, bbox_relative_to_absolute, bbox_to_relative')
print('  is_global_edit, is_full_image_bbox, load_json, save_json')
print('  ok  all smoke tests passed')


## §8.1 — Phase 1 Prompt Template (Read-Only)
**MUST be byte-identical to Phase 1 §8.1 and Phase 2 §8.1.**
Any change invalidates `PHASE1_TEMPLATE_HASH` and breaks conditioning alignment.

Phase 3 uses `build_messages()` at inference with `annotations=[]` (no ground-truth
regions available at runtime). The model predicts bbox from visual context alone.
Both generation and hidden-state extraction use this same no-annotation prompt.
This is a known train/inference distribution gap — see notebook header.

In [ ]:
# -- §8.1  Phase 1 prompt template -- BYTE-IDENTICAL to Phase 1 §8.1 -----------
#
# CRITICAL: Do NOT create a second implementation. Any change here invalidates
# PHASE1_TEMPLATE_HASH and means Phase 3 conditioning diverges from training.
#
# Phase 3 distribution gap (known, acknowledged in spec §7):
#   Training hidden states were extracted with full region lists in the prompt.
#   Inference extracts hidden states WITHOUT region lists (not available at runtime).
#   Both the VLM generation and hidden-state extraction use the same no-annotation
#   prompt -- they are internally consistent with each other.
#   The gap is between the training-time prompt distribution and inference-time.

import hashlib as _hashlib
import json as _json_mod

PHASE1_SYSTEM_PROMPT = (
    'You are an image editing assistant. '
    'Given a source image, a list of segmentation regions with bounding boxes, '
    'and a natural-language edit instruction, '
    'identify the target region and predict the edit operation as a JSON object.\n\n'
    'The segmentation regions are listed as:\n'
    '  - <class_name>: [x1, y1, x2, y2]\n'
    'where coordinates are in [0, 1000] relative scale '
    '(0 = top-left, 1000 = bottom-right).\n\n'
    'JSON schema:\n'
    '{\n'
    '  "edit_type": "<type>",          '
    '// one of: action, add, adjust, background, content,\n'
    '//         hybrid, reference, remove, replace, style, version\n'
    '  "bbox": [x1, y1, x2, y2],       '
    '// the bounding box of the target region in [0, 1000] coordinates\n'
    '//   (select the region from the list above that best matches the instruction)\n'
    '  "edit_description": "<text>"    '
    '// precise description of what to edit\n'
    '}\n\n'
    'Output ONLY valid JSON. No explanation, no markdown code fences.'
)

PHASE1_USER_REGIONS_HEADER = 'Segmentation regions:\n'
PHASE1_USER_PREFIX          = 'Edit instruction: '

_tpl_str = PHASE1_SYSTEM_PROMPT + '|SEP|' + PHASE1_USER_REGIONS_HEADER + '|SEP|' + PHASE1_USER_PREFIX
PHASE1_TEMPLATE_HASH = _hashlib.sha256(_tpl_str.encode()).hexdigest()[:16]
print(f'PHASE1_TEMPLATE_HASH = {PHASE1_TEMPLATE_HASH!r}')
print('  Must match Phase 2 hidden_states_meta.json["phase1_template_hash"] (checked in §20).')


def format_regions_text(annotations: list, seg_W: int, seg_H: int) -> str:
    '''Format segmentation regions as [0,1000] region list.
    Identical to Phase 1 §8.1. Returns empty string when annotations is empty.
    '''
    if not annotations:
        return ''
    lines = [PHASE1_USER_REGIONS_HEADER]
    for ann in annotations:
        class_name = (ann.get('class_name') or 'unknown').strip()
        raw_bbox   = ann.get('bbox') or []
        if len(raw_bbox) != 4:
            continue
        rel_bbox = bbox_to_relative(raw_bbox, seg_W, seg_H)
        lines.append(f'  - {class_name}: {rel_bbox}\n')
    return ''.join(lines)


def build_messages(
    source_image,
    instruction_text: str,
    annotations: list = None,
    seg_dims: tuple   = None,
) -> list:
    '''Build Qwen2.5-VL chat messages. Byte-identical to Phase 1 §8.1.

    Phase 3 calls with annotations=None -- region list omitted gracefully.
    The model predicts bbox from visual understanding + fine-tuned weights.
    '''
    if isinstance(source_image, (str, Path)):
        source_image = Image.open(source_image).convert('RGB')
    elif not isinstance(source_image, Image.Image):
        raise TypeError(f'source_image must be PIL Image or Path, got {type(source_image)}')

    seg_W, seg_H = seg_dims if seg_dims else (1, 1)
    if seg_dims is None and annotations:
        log.warning(
            'build_messages: annotations provided but seg_dims is None. '
            'Region bboxes will be wrong. Always pass seg_dims when annotations present.'
        )

    regions_text = format_regions_text(annotations or [], seg_W, seg_H)
    user_text    = regions_text + PHASE1_USER_PREFIX + instruction_text

    return [
        {'role': 'system', 'content': PHASE1_SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': source_image},
                {'type': 'text',  'text':  user_text},
            ],
        },
    ]


# -- Smoke tests ----------------------------------------------------------
_t = Image.new('RGB', (32, 32), (100, 100, 100))
_m_no_ann = build_messages(_t, 'remove the car')
assert PHASE1_USER_REGIONS_HEADER not in _m_no_ann[1]['content'][1]['text'], (
    'No-annotation path must NOT emit region header'
)
assert PHASE1_USER_PREFIX in _m_no_ann[1]['content'][1]['text']
_m_ann = build_messages(
    _t, 'remove the car',
    annotations=[{'class_name': 'car', 'bbox': [100, 100, 400, 300]}],
    seg_dims=(512, 512),
)
assert PHASE1_USER_REGIONS_HEADER in _m_ann[1]['content'][1]['text']
print('  ok  build_messages no-annotation path (Phase 3 inference path)')
print('  ok  build_messages annotation path')
print(f'\nPhase 3 user turn (no annotations):')
print(_m_no_ann[1]['content'][1]['text'])


### §2.3 — SAM2 Installation & Model Loading

### §21.1 — SAM2 Install (run once per runtime) + Checkpoint Download

In [ ]:
# -- §21.1  SAM2 install from GitHub + checkpoint download to Drive -----------
#
# SAM2 install fragility (known Colab issue):
#   - The sam2 Python package is NOT on PyPI; must be installed from source each runtime.
#   - The checkpoint (1.2 GB) is cached to Drive; subsequent restarts skip download.
#   - If 'from sam2.build_sam import build_sam2' fails, re-run this cell.
#
# Checkpoint URL: SAM2.1 Hiera Large (092824 release).
# The Large model is chosen for accuracy; Small (sam2.1_hiera_small.pt) is faster
# but less precise for complex object boundaries.

import subprocess, sys, os
from pathlib import Path

#assert '_PREFLIGHT_PASSED' in dir() and _PREFLIGHT_PASSED, (
#    '§20 preflight check must pass before §21. Run §20.1 first.'
#)

_SAM2_GITHUB_DIR = '/content/sam2'

# -- Install SAM2 from GitHub (every runtime — /content is ephemeral) ------
if not os.path.exists(_SAM2_GITHUB_DIR):
    print('Cloning SAM2 from GitHub...')
    _r = subprocess.run(
        ['git', 'clone', '--quiet',
         'https://github.com/facebookresearch/sam2.git', _SAM2_GITHUB_DIR],
        capture_output=True, text=True
    )
    if _r.returncode != 0:
        raise RuntimeError(f'git clone sam2 failed:\n{_r.stderr[-500:]}')
    print('  ok  git clone sam2')
else:
    print(f'  ok  {_SAM2_GITHUB_DIR} already exists')

_r2 = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', _SAM2_GITHUB_DIR],
    capture_output=True, text=True
)
if _r2.returncode != 0:
    raise RuntimeError(f'pip install sam2 failed:\n{_r2.stderr[-500:]}')
print('  ok  sam2 package installed from source')

# -- Verify import (sys.path guard -- see §1.2 comments) ------------------
# SAM2's build_sam.py raises RuntimeError under TWO independent conditions:
#   1. os.getcwd() equals the parent of the cloned repo (/content in Colab).
#      The check compares os.getcwd() to os.path.dirname(os.path.dirname(__file__))
#      which resolves to /content when the repo lives at /content/sam2/.
#   2. /content (or '' which also resolves to CWD) is in sys.path, allowing
#      'sam2' to resolve to the repo root as a namespace package rather than the
#      installed package at /content/sam2/sam2/.
# BOTH must be fixed. The previous fix only modified sys.path — this is why
# the RuntimeError persisted. The correct fix requires:
#   (a) os.chdir('/root') so os.getcwd() no longer matches /content, AND
#   (b) filter sys.path to remove '' and /content, AND
#   (c) clear any stale sys.modules['sam2'] cache from a prior failed import.
import sys as _sys_sam2, os as _os_sam2

_saved_cwd     = _os_sam2.getcwd()           # /content in Colab
_saved_syspath = _sys_sam2.path[:]

_SAM2_REPO_PARENT = '/content'
_sys_sam2.path = [
    p for p in _sys_sam2.path
    if p != ''                                              # remove '' (= CWD ref)
    and _os_sam2.path.normpath(p) != _os_sam2.path.normpath(_SAM2_REPO_PARENT)
]

# Clear stale sam2 entries a prior failed import may have cached. Without this,
# Python reuses the broken namespace-package entry even after sys.path is fixed.
for _k in [k for k in list(_sys_sam2.modules) if k == 'sam2' or k.startswith('sam2.')]:
    del _sys_sam2.modules[_k]

# Change CWD to /root — no sam2 subdir there, so build_sam.py's CWD check passes.
_os_sam2.chdir('/root')
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
finally:
    _os_sam2.chdir(_saved_cwd)
    _sys_sam2.path = _saved_syspath   # restore regardless of success/failure
try:
    _ = build_sam2
    print('  ok  sam2 importable after install')
except (ImportError, RuntimeError, NameError) as _e:
    raise RuntimeError(
        f'SAM2 import still fails after install: {_e}\n'
        'Try: Runtime -> Restart session, then re-run this cell (§21.1)'
    )

# -- Download checkpoint to Drive (skip if already cached) -----------------
_SAM2_CKPT_URL = (
    'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'
)

if CFG.sam2_checkpoint.exists():
    _size_mb = CFG.sam2_checkpoint.stat().st_size / 1e6
    print(f'  ok  SAM2 checkpoint cached on Drive ({_size_mb:.0f} MB): {CFG.sam2_checkpoint}')
else:
    print(f'Downloading SAM2 checkpoint (~1.2 GB) to Drive...')
    print(f'  URL       : {_SAM2_CKPT_URL}')
    print(f'  Saving to : {CFG.sam2_checkpoint}')
    CFG.sam2_dir.mkdir(parents=True, exist_ok=True)
    _r3 = subprocess.run(
        ['wget', '-q', '-O', str(CFG.sam2_checkpoint), _SAM2_CKPT_URL],
        capture_output=True, text=True
    )
    if _r3.returncode != 0:
        CFG.sam2_checkpoint.unlink(missing_ok=True)  # clean partial download
        raise RuntimeError(f'SAM2 checkpoint download failed:\n{_r3.stderr[-300:]}')
    _size_mb = CFG.sam2_checkpoint.stat().st_size / 1e6
    if _size_mb < 100:
        CFG.sam2_checkpoint.unlink(missing_ok=True)
        raise RuntimeError(
            f'Downloaded file is too small ({_size_mb:.1f} MB) -- likely a 404 or partial download.\n'
            f'Check URL: {_SAM2_CKPT_URL}'
        )
    print(f'  ok  SAM2 checkpoint downloaded ({_size_mb:.0f} MB)')

print('\n§21.1 complete. SAM2 ready.')


### §21.2 — Load SAM2 Model & Predictor

In [ ]:
# -- §21.2  Load SAM2 model and image predictor --------------------------------
#
# ROOT CAUSE OF MissingConfigException:
#   build_sam2 uses Hydra with pkg:// config resolution. On Python 3.12 with
#   `pip install -e` editable installs, importlib.resources fails to traverse
#   package subdirectories, so Hydra can't find the YAML even if it's on disk.
#
# FIX — use initialize_config_dir (absolute filesystem path):
#   Pre-initialise Hydra with the absolute path to the SAM2 configs directory
#   BEFORE calling build_sam2. build_sam2 skips its own Hydra init when Hydra
#   is already initialised, so compose() resolves configs from the filesystem.
#
# Config directory discovery:
#   The SAM2 repo layout has configs at the REPO ROOT (/content/sam2/configs/)
#   in some releases, and inside the Python package (/content/sam2/sam2/configs/)
#   in others. We probe both directories.
#
# Config FILENAME discovery:
#   The Large config is named "sam2.1_hiera_l.yaml" in the current upstream
#   repo, but past forks/releases used "sam2.1_hiera_large.yaml". We probe
#   both variants and use whichever exists on disk, regardless of which name
#   the user has in CFG. This makes the cell robust to either spelling.

import os, torch
from pathlib import Path

# -- sys.path / CWD guard for SAM2 import ----------------------------------
# build_sam.py raises RuntimeError if os.getcwd() == /content (parent of the
# cloned repo). Fix: chdir to /root for the import only, then restore.
import sys as _sys_sam2, os as _os_sam2

_saved_cwd     = _os_sam2.getcwd()
_saved_syspath = _sys_sam2.path[:]
_SAM2_REPO_PARENT = '/content'

_sys_sam2.path = [
    p for p in _sys_sam2.path
    if p != ''
    and _os_sam2.path.normpath(p) != _os_sam2.path.normpath(_SAM2_REPO_PARENT)
]
for _k in [k for k in list(_sys_sam2.modules) if k == 'sam2' or k.startswith('sam2.')]:
    del _sys_sam2.modules[_k]

_os_sam2.chdir('/root')
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
finally:
    _os_sam2.chdir(_saved_cwd)
    _sys_sam2.path = _saved_syspath

#assert '_PREFLIGHT_PASSED' in dir() and _PREFLIGHT_PASSED

# -- Locate SAM2 configs directory -----------------------------------------
# The SAM2 repo places configs either at the repo root or inside the package.
# Probe both and use whichever contains the target YAML.
from hydra import initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

_SAM2_GITHUB_DIR   = '/content/sam2'
_SAM2_CONFIG_NAME  = CFG.phase3_sam2_model_cfg   # e.g. "sam2.1/sam2.1_hiera_l"

# Build candidate (config_dir, config_name_without_yaml) pairs.
# The SAM2 repo places configs at one of two locations depending on release:
#   - /content/sam2/configs/             (older repo-root layout)
#   - /content/sam2/sam2/configs/        (current package-internal layout)
# AND the Large config is named "sam2.1_hiera_l.yaml" in the current repo, but
# some past releases / forks used "sam2.1_hiera_large.yaml". To be robust we
# probe BOTH directories AND BOTH filename variants, and resolve to whatever
# actually exists on disk.
_dir_candidates = [
    os.path.join(_SAM2_GITHUB_DIR, 'sam2', 'configs'),  # current layout — try first
    os.path.join(_SAM2_GITHUB_DIR, 'configs'),          # legacy repo-root layout
]

# Filename variants: try the configured name first, then the "_l" <-> "_large"
# alternative. This handles both directions (CFG="_large", repo has "_l", and
# vice versa).
_name_candidates = [_SAM2_CONFIG_NAME]
if _SAM2_CONFIG_NAME.endswith('_large'):
    _name_candidates.append(_SAM2_CONFIG_NAME[:-len('_large')] + '_l')
elif _SAM2_CONFIG_NAME.endswith('_l'):
    _name_candidates.append(_SAM2_CONFIG_NAME[:-len('_l')] + '_large')

_SAM2_CONFIGS_DIR  = None
_SAM2_CONFIG_NAME_RESOLVED = None
_searched = []
for _cand_dir in _dir_candidates:
    for _cand_name in _name_candidates:
        _yaml_path = os.path.join(_cand_dir, _cand_name + '.yaml')
        _searched.append(_yaml_path)
        if Path(_yaml_path).exists():
            _SAM2_CONFIGS_DIR = os.path.abspath(_cand_dir)
            _SAM2_CONFIG_NAME_RESOLVED = _cand_name
            break
    if _SAM2_CONFIGS_DIR is not None:
        break

if _SAM2_CONFIGS_DIR is None:
    _checked = '\n  '.join(_searched)
    raise FileNotFoundError(
        f"SAM2 config YAML for '{_SAM2_CONFIG_NAME}' not found.\n"
        f"Searched:\n  {_checked}\n"
        f"Re-run §21.1 to re-clone the SAM2 repo, then retry §21.2."
    )

if _SAM2_CONFIG_NAME_RESOLVED != _SAM2_CONFIG_NAME:
    print(f"  note  CFG specified '{_SAM2_CONFIG_NAME}' but repo has "
          f"'{_SAM2_CONFIG_NAME_RESOLVED}.yaml' — using the repo filename.")
_SAM2_CONFIG_NAME = _SAM2_CONFIG_NAME_RESOLVED  # use the on-disk name

print(f'Loading SAM2 model...')
print(f'  Config dir : {_SAM2_CONFIGS_DIR}')
print(f'  Config     : {_SAM2_CONFIG_NAME}')
print(f'  Checkpoint : {CFG.sam2_checkpoint}')

# Clear stale Hydra global state (critical for re-runs)
GlobalHydra.instance().clear()

# Pre-initialise Hydra with the absolute filesystem path so build_sam2's
# pkg:// resolver is bypassed entirely.
with initialize_config_dir(config_dir=_SAM2_CONFIGS_DIR, version_base="1.2"):
    sam2_model = build_sam2(
        config_file = _SAM2_CONFIG_NAME,
        ckpt_path   = str(CFG.sam2_checkpoint),
        device      = DEVICE,
    )
sam2_model.eval()
print(f'  ok  SAM2 model loaded ({type(sam2_model).__name__})')

sam2_predictor = SAM2ImagePredictor(sam2_model)
print(f'  ok  SAM2ImagePredictor ready')

# -- SAM2 smoke test -------------------------------------------------------
import numpy as np
_smoke_img_np = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
sam2_predictor.set_image(_smoke_img_np)

_smoke_box = np.array([[0, 0, 64, 64]], dtype=np.float32)
with torch.inference_mode():
    _masks, _scores, _logits = sam2_predictor.predict(
        point_coords     = None,
        point_labels     = None,
        box              = _smoke_box,
        multimask_output = False,
    )

assert _masks.ndim == 3 and _masks.shape[0] == 1, (
    f'SAM2 mask shape unexpected: {_masks.shape}. Expected (1, H, W).'
)
assert _masks.shape[1:] == (64, 64), (
    f'SAM2 mask spatial dims {_masks.shape[1:]} != image dims (64,64)'
)
assert _scores.shape == (1,), f'Scores shape {_scores.shape} != (1,)'
print(f'  ok  SAM2 smoke test: mask shape={tuple(_masks.shape)}, score={_scores[0]:.3f}')
print(f'\n§21.2 SAM2 loaded and verified.')
print(f'  sam2_model:     {type(sam2_model).__name__}')
print(f'  sam2_predictor: {type(sam2_predictor).__name__}')


### §2.4 — VLM Loading (Phase 1 Fine-Tuned)

In [ ]:
# -- §22.1  Load Phase 1 fine-tuned VLM (base + LoRA adapter) -----------------
#
# Loading strategy:
#   1. Load base Qwen2.5-VL-3B-Instruct with 4-bit NF4 quantization
#      (same quantization config as Phase 1 training)
#   2. Attach LoRA adapter from CFG.ckpt_phase1 using PeftModel.from_pretrained
#   3. Set model to eval mode -- inference only, no gradient computation
#
# process_vision_info: extracts PIL images from Qwen message dicts.
# qwen-vl-utils is the primary source; inline fallback if not installed.

import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
)
from peft import PeftModel

#assert '_PREFLIGHT_PASSED' in dir() and _PREFLIGHT_PASSED

# -- process_vision_info: primary from qwen_vl_utils, fallback inline ------
try:
    from qwen_vl_utils import process_vision_info
    print('  ok  Using qwen_vl_utils.process_vision_info')
except ImportError:
    print('  WARN  qwen_vl_utils not available -- using built-in fallback')

    def process_vision_info(messages: list):
        '''Fallback: extract PIL images from Qwen2.5-VL message content dicts.
        Identical to Phase 1 §9.1 fallback implementation.
        '''
        images = []
        for msg in messages:
            content = msg.get('content', [])
            if not isinstance(content, list):
                continue
            for item in content:
                if not isinstance(item, dict) or item.get('type') != 'image':
                    continue
                img = item.get('image')
                if isinstance(img, Image.Image):
                    images.append(img)
                elif isinstance(img, (str, Path)):
                    images.append(Image.open(img).convert('RGB'))
                else:
                    raise TypeError(
                        f'Unsupported image type {type(img)}. Pass PIL.Image.Image.'
                    )
        return images, []

# -- Quantization config (identical to Phase 1) ----------------------------
_bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

print(f'Loading base model: {CFG.vlm_model_id}')
print('  (First run downloads ~2 GB to HF cache; subsequent runs are instant)')
_base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG.vlm_model_id,
    quantization_config = _bnb_config,
    device_map          = {'': DEVICE},
    torch_dtype         = torch.bfloat16,
    # max_pixels caps image patch count to avoid OOM on high-res images
    # Must match Phase 1 training config (phase1_max_img_pixels)
)
print(f'  ok  base model loaded')

# -- Attach LoRA adapter from Phase 1 checkpoint ---------------------------
print(f'\nLoading Phase 1 LoRA adapter from: {CFG.ckpt_phase1}')
vlm_model = PeftModel.from_pretrained(
    _base_model,
    str(CFG.ckpt_phase1),
    is_trainable = False,   # inference only
)
vlm_model.eval()
print(f'  ok  LoRA adapter attached')

# Verify hidden_size matches CFG.vlm_hidden_dim.
#
# Qwen2.5-VL config layout has changed across transformers versions:
#   * Older releases expose `config.hidden_size` directly on Qwen2_5_VLConfig.
#   * Newer releases (current Colab default) split the config into
#     `config.text_config` and `config.vision_config`; the LM hidden dim
#     lives at `config.text_config.hidden_size` and the top-level attribute
#     is gone, raising AttributeError if accessed naively.
#
# We probe a few well-known locations and fail fast with full diagnostic
# context if none of them resolves -- never silently fall back to CFG.
def _resolve_vlm_hidden_size(mdl) -> int:
    cfg = mdl.config
    candidates = [
        ('config.hidden_size',              lambda c: getattr(c, 'hidden_size', None)),
        ('config.text_config.hidden_size',  lambda c: getattr(getattr(c, 'text_config', None),  'hidden_size', None)),
        ('config.llm_config.hidden_size',   lambda c: getattr(getattr(c, 'llm_config',  None),  'hidden_size', None)),
    ]
    for label, getter in candidates:
        val = getter(cfg)
        if isinstance(val, int) and val > 0:
            print(f'  ok  resolved hidden_size via {label} = {val}')
            return val
    # Fail loudly with diagnostics rather than guessing.
    raise AttributeError(
        f'Could not resolve hidden_size on {type(cfg).__name__}. '
        f'Tried: {[l for l,_ in candidates]}. '
        f'Available top-level attrs (filtered): '
        f'{[a for a in dir(cfg) if not a.startswith("_") and "config" in a.lower() or "hidden" in a.lower()][:20]}'
    )

_actual_hidden = _resolve_vlm_hidden_size(vlm_model)
assert _actual_hidden == CFG.vlm_hidden_dim, (
    f'Model hidden_size={_actual_hidden} != CFG.vlm_hidden_dim={CFG.vlm_hidden_dim}. '
    f'Update CFG.vlm_hidden_dim to {_actual_hidden}.'
)
print(f'  ok  hidden_size={_actual_hidden} matches CFG.vlm_hidden_dim')

# -- Load processor (tokenizer + image processor) -------------------------
# Load from checkpoint dir first (processor was saved with adapter in Phase 1 §10.2).
# Fall back to hub if not present (older Phase 1 checkpoints).
_proc_path = CFG.ckpt_phase1
if not (_proc_path / 'tokenizer_config.json').exists():
    _proc_path = CFG.vlm_model_id
    print(f'  WARN  No processor in checkpoint -- loading from hub: {CFG.vlm_model_id}')

vlm_processor = AutoProcessor.from_pretrained(str(_proc_path))
print(f'  ok  processor loaded from {_proc_path}')

# Freeze all VLM weights (LoRA included) -- no gradient needed at inference
for _p in vlm_model.parameters():
    _p.requires_grad_(False)

print(f'\n§22.1  VLM ready for inference.')
print(f'  Model: {type(vlm_model).__name__}')
print(f'  Quantization: 4-bit NF4 (bfloat16 compute)')
print(f'  Device: {next(vlm_model.parameters()).device}')


### §2.5 — GroundingDINO Loader (v3.0)

In [ ]:
# -- §22.5  Load GroundingDINO (v3.0) ----------------------------------------
#
# Loads `IDEA-Research/grounding-dino-tiny` from the HF hub via
# AutoModelForZeroShotObjectDetection + AutoProcessor. We deliberately use the
# Transformers port instead of the standalone `groundingdino-py` package:
#   * No CUDA-extension build — works on Colab GPUs that lack matching nvcc.
#   * Uses the same AutoProcessor pattern as the rest of the notebook.
#   * Tiny variant is ~170 MB and runs in < 200 ms per image on an A100.
#
# We freeze all weights and load in fp16 to share VRAM with VLM + SAM2 + SD.
# The `run_grounding_dino` helper below is the only call site.

import torch
import numpy as np
from PIL import Image
from typing import List, Tuple
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

#assert "_PREFLIGHT_PASSED" in dir() and _PREFLIGHT_PASSED

print(f"Loading GroundingDINO: {CFG.grounding_model_id}")
print("  (first run downloads ~170 MB to HF cache)")

# Processor handles both image preprocessing and tokenizing the text prompt.
grounding_processor = AutoProcessor.from_pretrained(CFG.grounding_model_id)

# Load in fp16 directly on the CUDA device. GroundingDINO ships a single-call
# API; we don't need autocast acrobatics.
#grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
#    CFG.grounding_model_id,
#    torch_dtype=torch.float16,
#).to(DEVICE).eval()

# NOTE: We deliberately load GroundingDINO in fp32, not fp16.
# The HF transformers port has a known dtype-mixing bug in the text branch:
# `text_enhancer_layer.with_pos_embed` adds fp32 sinusoidal position
# embeddings to fp16 hidden states, producing fp32 queries that then hit
# fp16 Linear weights -> "mat1 and mat2 must have the same dtype" RuntimeError.
# The tiny variant is ~700 MB in fp32, which is acceptable alongside the
# rest of the pipeline. Do not "optimize" this back to fp16 without first
# confirming the upstream bug is fixed.
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    CFG.grounding_model_id,
    torch_dtype=torch.float32,
).to(DEVICE).eval()

for _p in grounding_model.parameters():
    _p.requires_grad_(False)

print(f"  ok  GroundingDINO loaded ({type(grounding_model).__name__})")
print(f"      device={next(grounding_model.parameters()).device}, "
      f"dtype={next(grounding_model.parameters()).dtype}")


def _format_grounding_prompt(phrase: str) -> str:
    """GroundingDINO expects a period-terminated, lowercased prompt.

    The HF docs explicitly require this format — without the trailing period
    the model's text-token alignment regresses sharply. We also strip leading
    articles so the model sees the noun head ("the motorcycle" -> "motorcycle.").
    Multi-noun queries can be passed as "motorcycle. text. cake.".
    """
    p = (phrase or "").strip().lower()
    if not p:
        return ""
    # Strip a single leading article — pre-trained DINO is more robust to bare nouns.
    for _art in ("the ", "a ", "an "):
        if p.startswith(_art):
            p = p[len(_art):]
            break
    if not p.endswith("."):
        p = p + "."
    return p


def run_grounding_dino(
    image: Image.Image,
    phrase: str,
) -> List[Tuple[List[float], float]]:
    """Run GroundingDINO and return [(bbox_xyxy_abs, score), ...] sorted by score.

    Args:
        image  : PIL.Image (RGB).
        phrase : free-form text describing the target object(s).

    Returns:
        List of (bbox_xyxy_abs_pixels, score) tuples, max len = grounding_max_boxes.
        Empty list if no detections cleared the threshold.
    """
    prompt = _format_grounding_prompt(phrase)
    if not prompt:
        log.info("run_grounding_dino: empty prompt — skipping detection")
        return []

    img_rgb = image.convert("RGB")
    W, H    = img_rgb.size

    inputs = grounding_processor(
        images=img_rgb,
        text=prompt,
        return_tensors="pt",
    ).to(DEVICE)

    # GroundingDINO expects fp16 image features + fp32 attention masks; the
    # processor outputs fp32 everywhere, so we cast pixel_values only.
    #if "pixel_values" in inputs:
    #    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    # Model is loaded in fp32 (see §22.5 note on the text-branch dtype bug);
    # processor already returns fp32 tensors, so no explicit casting needed.
    # We keep this assertion to fail fast if anyone re-introduces fp16 loading
    # without addressing the text-branch mismatch.
    assert next(grounding_model.parameters()).dtype == torch.float32, (
        "GroundingDINO must be loaded in fp32; see §22.5 comment for why."
    )

    with torch.no_grad():
        outputs = grounding_model(**inputs)

    # post_process_grounded_object_detection returns boxes in xyxy abs coords
    # for the ORIGINAL image size when target_sizes is provided.
    target_sizes = torch.tensor([[H, W]], device=DEVICE)
    #results = grounding_processor.post_process_grounded_object_detection(
    #    outputs,
    #    inputs["input_ids"],
    #    box_threshold=CFG.grounding_box_threshold,
    #    text_threshold=CFG.grounding_text_threshold,
    #    target_sizes=target_sizes,
   # )[0]
   # NOTE: transformers renamed `box_threshold` -> `threshold` in recent
    # versions of GroundingDinoProcessor.post_process_grounded_object_detection.
    # `text_threshold` kept its name. We pass via the new name; if you pin
    # an older transformers, swap back to `box_threshold=`.
    results = grounding_processor.post_process_grounded_object_detection(
        outputs,
        inputs["input_ids"],
        threshold=CFG.grounding_box_threshold,
        text_threshold=CFG.grounding_text_threshold,
        target_sizes=target_sizes,
    )[0]

    boxes  = results["boxes"].detach().cpu().numpy().tolist()   # list of [x1,y1,x2,y2]
    scores = results["scores"].detach().cpu().numpy().tolist()  # list of floats

    if not boxes:
        log.info(
            f"run_grounding_dino: 0 detections for prompt {prompt!r} "
            f"(box_thr={CFG.grounding_box_threshold}, "
            f"text_thr={CFG.grounding_text_threshold})"
        )
        return []

    # Pair, sort by descending score, cap to grounding_max_boxes.
    paired = sorted(zip(boxes, scores), key=lambda bs: -bs[1])
    paired = paired[: CFG.grounding_max_boxes]

    log.info(
        f"run_grounding_dino: prompt={prompt!r} -> {len(paired)} boxes "
        f"(top score={paired[0][1]:.3f})"
    )
    return paired


# -- Smoke test ------------------------------------------------------------
# We test on a 256x256 noise image. With nothing recognizable, we expect the
# model to return zero detections — which is the correct behaviour and proves
# the post-processing call is wired up.
_smoke_img   = Image.fromarray(
    np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8), mode="RGB"
)
_smoke_boxes = run_grounding_dino(_smoke_img, "a cat")
print(f"  ok  GroundingDINO smoke test on noise image: "
      f"{len(_smoke_boxes)} detections (expected 0 for noise)")
print(f"\n§22.5 GroundingDINO ready for inference.")


### §2.6 — SD Pipeline Components (Phase 2 Checkpoint)

In [ ]:
# -- §23.1  Load SD inpainting components (VAE, UNet, CLIP, DDIMScheduler) ----
# (v1.9: added fail-fast scheduler parity assertions against Phase 2's effective
#        config. Architecture itself is unchanged from v1.8 — same repo, same
#        9-channel inpainting UNet, same CLIP/VAE.)
#
# Same component loading as Phase 2 §14.1 — identical arguments.
# DDIM replaces DDPM for inference (fewer steps: 20 vs 1000).
#
# CRITICAL: load UNet as fp32 first, then cast LoRA + UNet to fp16 after
# PEFT attach. PeftModel wraps the UNet and is not a UNet2DConditionModel
# instance, so from_pretrained(..., torch_dtype=fp16) skips the cast.
# Explicit cast to fp16 is done in §23.2 after PEFT attach.

import torch
from diffusers import (
    AutoencoderKL, UNet2DConditionModel, DDIMScheduler,
    StableDiffusionInpaintPipeline,
)
from transformers import CLIPTextModel, CLIPTokenizer

#assert '_PREFLIGHT_PASSED' in dir() and _PREFLIGHT_PASSED

print(f'Loading SD inpainting components from: {CFG.sd_model_id}')

# VAE -- frozen, fp32 (stability for encode/decode)
vae = AutoencoderKL.from_pretrained(
    CFG.sd_model_id, subfolder='vae', torch_dtype=torch.float32
)
vae.eval()
vae.requires_grad_(False)
vae = vae.to(DEVICE)
print(f'  ok  VAE loaded')

# UNet -- 9-channel inpainting variant (channels: noisy_tgt | mask | masked_src)
# Load fp32; will be cast to fp16 after LoRA attach in §23.2
unet = UNet2DConditionModel.from_pretrained(
    CFG.sd_model_id, subfolder='unet', torch_dtype=torch.float32
)
assert unet.config.in_channels == 9, (
    f'Expected UNet in_channels=9 (inpainting), got {unet.config.in_channels}.\n'
    f'Use runwayml/stable-diffusion-inpainting, not base SD 1.5.'
)
assert unet.config.cross_attention_dim == CFG.sd_cross_attn_dim, (
    f'UNet cross_attention_dim={unet.config.cross_attention_dim} '
    f'!= CFG.sd_cross_attn_dim={CFG.sd_cross_attn_dim}'
)
unet = unet.to(DEVICE)
print(f'  ok  UNet loaded: in_channels={unet.config.in_channels}, '
      f'cross_attention_dim={unet.config.cross_attention_dim}')

# CLIP text encoder + tokenizer -- frozen
sd_tokenizer = CLIPTokenizer.from_pretrained(CFG.sd_model_id, subfolder='tokenizer')
clip = CLIPTextModel.from_pretrained(
    CFG.sd_model_id, subfolder='text_encoder', torch_dtype=torch.float32
)
clip.eval()
clip.requires_grad_(False)
clip = clip.to(DEVICE)
print(f'  ok  CLIP loaded: hidden_size={clip.config.hidden_size}')

# DDIM scheduler -- faster inference (20 steps vs DDPM 1000)
#
# Train/inference parity note (verified against phase2_sd_bridge_training_v1_5):
#   Phase 2 constructs the training scheduler with
#       DDPMScheduler.from_pretrained(CFG.sd_model_id, subfolder='scheduler')
#   and only overrides num_train_timesteps. Despite CFG.phase2_beta_schedule='linear'
#   being defined, that constant is NEVER applied to the scheduler in Phase 2,
#   so the UNet was effectively trained against the runwayml inpainting repo's
#   shipped beta_schedule (= 'scaled_linear'). To preserve train/inference parity
#   we MUST load Phase 3's DDIM scheduler from the same repo+subfolder and assert
#   the betas match what training saw. The assertions below fail-fast on any drift
#   (e.g. someone editing the scheduler config, or the upstream repo changing).
noise_scheduler = DDIMScheduler.from_pretrained(CFG.sd_model_id, subfolder='scheduler')

# --- Fail-fast parity checks against Phase 2 effective config -------------------
# Phase 2 inherited these values from the runwayml/stable-diffusion-inpainting
# scheduler config; if any of them changes here, denoising at inference will
# operate on a different noise schedule than the UNet was trained against.
_EXPECTED_BETA_SCHEDULE = 'scaled_linear'   # inherited by Phase 2 from the repo
_EXPECTED_BETA_START   = 0.00085            # SD 1.5 family default
_EXPECTED_BETA_END     = 0.012              # SD 1.5 family default
_EXPECTED_NUM_TRAIN_TIMESTEPS = 1000        # Phase 2 explicitly sets this

_bs  = noise_scheduler.config.beta_schedule
_b0  = float(noise_scheduler.config.beta_start)
_b1  = float(noise_scheduler.config.beta_end)
_nts = int(noise_scheduler.config.num_train_timesteps)

assert _bs == _EXPECTED_BETA_SCHEDULE, (
    f'beta_schedule mismatch vs Phase 2: got {_bs!r}, expected {_EXPECTED_BETA_SCHEDULE!r}. '
    f'Training UNet against one schedule and sampling with another produces degraded outputs.'
)
assert abs(_b0 - _EXPECTED_BETA_START) < 1e-9, (
    f'beta_start mismatch vs Phase 2: got {_b0}, expected {_EXPECTED_BETA_START}'
)
assert abs(_b1 - _EXPECTED_BETA_END) < 1e-9, (
    f'beta_end mismatch vs Phase 2: got {_b1}, expected {_EXPECTED_BETA_END}'
)
assert _nts == _EXPECTED_NUM_TRAIN_TIMESTEPS, (
    f'num_train_timesteps mismatch vs Phase 2: got {_nts}, expected {_EXPECTED_NUM_TRAIN_TIMESTEPS}'
)

# Inference-only step count; does NOT change the underlying noise schedule.
noise_scheduler.set_timesteps(CFG.phase3_num_inference_steps)

print(f'  ok  DDIMScheduler: {CFG.phase3_num_inference_steps} inference steps, '
      f'beta_schedule={_bs}, beta_start={_b0}, beta_end={_b1}, '
      f'num_train_timesteps={_nts}  (parity vs Phase 2 verified)')

print(f'\n§23.1 SD components loaded. Proceed to §23.2 to load trained adapters.')


In [ ]:
# -- §23.2  Load VLMProjectionAdapter and UNet LoRA from Phase 2 checkpoint ---
#
# VLMProjectionAdapter architecture: byte-identical to Phase 2 §14.3.
# Any change here means the loaded weights will mismatch the checkpoint layout.
#
# UNet LoRA loading:
#   Preferred: PeftModel.from_pretrained from unet_lora_hf/ (PEFT format)
#   Fallback:  load_state_dict from unet_lora.pt (manual tensor save)
#              (used when unet.save_pretrained failed in Phase 2 §18)

import torch
import torch.nn as nn
from peft import PeftModel

# -- VLMProjectionAdapter definition (byte-identical to Phase 2 §14.3) -----
class VLMProjectionAdapter(nn.Module):
    '''Projects VLM mean-pooled hidden state to SD cross-attention embedding.

    Input:  (batch, vlm_dim=2048)     float32
    Output: (batch, 1, sd_dim=768)    float32

    Concatenated with CLIP text embeddings (batch,77,768):
        combined = cat([clip_embeds, vlm_proj], dim=1)  -> (batch, 78, 768)
    '''

    def __init__(self, vlm_dim: int = None, sd_dim: int = None):
        super().__init__()
        vlm_dim = vlm_dim or CFG.vlm_hidden_dim        # 2048
        sd_dim  = sd_dim  or CFG.sd_cross_attn_dim     # 768
        intermediate_dim = sd_dim * 2                   # 1536

        self.proj = nn.Sequential(
            nn.Linear(vlm_dim, intermediate_dim, bias=True),
            nn.LayerNorm(intermediate_dim),
            nn.GELU(),
            nn.Linear(intermediate_dim, sd_dim, bias=True),
        )
        # Weight init: small normal (matches Phase 2 definition exactly)
        for layer in self.proj:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, std=0.02)
                nn.init.zeros_(layer.bias)

    def forward(self, vlm_hidden: torch.Tensor) -> torch.Tensor:
        assert vlm_hidden.ndim == 2, (
            f'VLMProjectionAdapter expects (batch, vlm_dim), got {vlm_hidden.shape}'
        )
        return self.proj(vlm_hidden).unsqueeze(1)   # (batch, 1, sd_dim)


# Instantiate and load Phase 2 weights
vlm_adapter = VLMProjectionAdapter().to(DEVICE)
vlm_adapter.load_state_dict(
    torch.load(CFG.ckpt_phase2_final / 'vlm_adapter.pt', map_location=DEVICE)
)
vlm_adapter.eval()
for _p in vlm_adapter.parameters():
    _p.requires_grad_(False)
print(f'  ok  VLMProjectionAdapter loaded from Phase 2 checkpoint')

# Shape check
_test_hs  = torch.randn(2, CFG.vlm_hidden_dim, device=DEVICE)
_test_out = vlm_adapter(_test_hs)
assert _test_out.shape == (2, 1, CFG.sd_cross_attn_dim), (
    f'VLMProjectionAdapter output {_test_out.shape} != (2, 1, {CFG.sd_cross_attn_dim})'
)
print(f'  ok  VLMProjectionAdapter shape: (2,{CFG.vlm_hidden_dim}) -> {tuple(_test_out.shape)}')

# -- UNet LoRA from Phase 2 checkpoint ------------------------------------
_p2_unet_hf = CFG.ckpt_phase2_final / 'unet_lora_hf'
_p2_unet_pt = CFG.ckpt_phase2_final / 'unet_lora.pt'

if _p2_unet_hf.exists():
    # PEFT format (preferred): wraps the UNet with PEFT LoRA
    unet = PeftModel.from_pretrained(unet, str(_p2_unet_hf), is_trainable=False)
    print(f'  ok  UNet LoRA loaded from PEFT format: {_p2_unet_hf}')
elif _p2_unet_pt.exists():
    # Manual fallback: load LoRA state dict into base UNet
    _lora_state = torch.load(_p2_unet_pt, map_location=DEVICE)
    _missing, _unexpected = unet.load_state_dict(_lora_state, strict=False)
    if _unexpected:
        log.warning(f'UNet LoRA: {len(_unexpected)} unexpected keys in state dict')
    print(f'  ok  UNet LoRA loaded from manual .pt: {_p2_unet_pt}')
    print(f'      LoRA keys loaded: {len(_lora_state)}')
else:
    raise FileNotFoundError(
        f'No UNet LoRA checkpoint found at:\n'
        f'  {_p2_unet_hf} (PEFT format)\n'
        f'  {_p2_unet_pt} (manual .pt)\n'
        'Re-run Phase 2 §18.2 to complete training.'
    )

unet.eval()
for _p in unet.parameters():
    _p.requires_grad_(False)

# -- Cast UNet to fp16 (PeftModel is not auto-cast by from_pretrained) -----
# This is the same fix as Phase 2 §19: PeftModel wraps UNet2DConditionModel,
# so from_pretrained dtype= does not apply. Cast explicitly here.
unet = unet.to(dtype=torch.float16)
print(f'  ok  UNet cast to fp16 (dtype={next(unet.parameters()).dtype})')

print(f'\n§23.2  Phase 2 adapters loaded and verified.')
print(f'  VLMProjectionAdapter: frozen, DEVICE={DEVICE}')
print(f'  UNet LoRA:            frozen, dtype=fp16')


In [ ]:
# -- §23.3  Conditioning pipeline -- CLIP + VLM projection ------------------
#
# SPEC REQUIREMENT: 'Protect against train-inference mismatch.'
# This function MUST be byte-identical to Phase 2 §16.
# Using do_cfg=True at inference and do_cfg=False during training.
#
# Inference CFG pattern (called from run_inpainting §27.1):
#   cond = build_combined_conditioning([instr], vlm_hs, device, do_cfg=True)
#   # -> (2, 78, 768): rows 0 = uncond, rows 1 = cond
#   noise_pred_2b = unet(unet_input_tiled, t, encoder_hidden_states=cond).sample
#   # -> (2, 4, H/8, W/8)
#   noise_pred_uncond, noise_pred_cond = noise_pred_2b.chunk(2)
#   noise_pred = noise_pred_uncond + scale * (noise_pred_cond - noise_pred_uncond)

import torch


def build_combined_conditioning(
    instructions: list,
    vlm_hiddens: torch.Tensor,
    device: str,
    do_cfg: bool = False,
) -> torch.Tensor:
    '''Build combined (CLIP + VLM) conditioning tensor. Byte-identical to Phase 2 §16.

    Args:
        instructions : List[str] of length batch
        vlm_hiddens  : (batch, vlm_hidden_dim) float32
        device       : 'cuda' or 'cpu'
        do_cfg       : If True, prepend unconditional branch (inference only).
                       MUST be False during training.

    Returns:
        Training   (do_cfg=False): (batch, 78, 768)
        Inference  (do_cfg=True) : (2*batch, 78, 768)
                                   rows 0..B-1 = unconditional
                                   rows B..2B-1 = conditional
    '''
    batch = len(instructions)
    assert vlm_hiddens.shape == (batch, CFG.vlm_hidden_dim), (
        f'vlm_hiddens shape {vlm_hiddens.shape} != ({batch}, {CFG.vlm_hidden_dim})'
    )

    # CLIP text encoding
    clip_tokens = sd_tokenizer(
        instructions,
        padding        = 'max_length',
        truncation     = True,
        max_length     = 77,
        return_tensors = 'pt',
    ).to(device)
    with torch.no_grad():
        clip_embeds = clip(**clip_tokens).last_hidden_state  # (batch, 77, 768)

    # VLM projection
    vlm_proj = vlm_adapter(vlm_hiddens.to(device).float())   # (batch, 1, 768)

    # Conditional branch
    cond = torch.cat([clip_embeds, vlm_proj], dim=1)          # (batch, 78, 768)

    if not do_cfg:
        return cond

    # Unconditional branch (inference CFG only)
    uncond_tokens = sd_tokenizer(
        [''] * batch,
        padding        = 'max_length',
        truncation     = True,
        max_length     = 77,
        return_tensors = 'pt',
    ).to(device)
    with torch.no_grad():
        uncond_clip = uncond_tokens
        uncond_clip = clip(**uncond_tokens).last_hidden_state  # (batch, 77, 768)

    # Null VLM token: zeros -> no VLM-specific signal in unconditional branch
    null_vlm = torch.zeros(batch, 1, CFG.sd_cross_attn_dim, device=device)
    uncond   = torch.cat([uncond_clip, null_vlm], dim=1)       # (batch, 78, 768)

    # Stack: [uncond, cond] along batch dimension
    return torch.cat([uncond, cond], dim=0)                    # (2*batch, 78, 768)


# -- Smoke test -----------------------------------------------------------
_test_instr  = ['change the shirt to blue', 'remove the car']
_test_vlm_hs = torch.randn(2, CFG.vlm_hidden_dim, device=DEVICE)

_cond_train  = build_combined_conditioning(_test_instr, _test_vlm_hs, DEVICE, do_cfg=False)
assert _cond_train.shape == (2, 78, CFG.sd_cross_attn_dim), (
    f'Training cond {_cond_train.shape} != (2,78,{CFG.sd_cross_attn_dim})'
)
print(f'  ok  training cond:   {tuple(_cond_train.shape)}')

_cond_infer  = build_combined_conditioning(_test_instr, _test_vlm_hs, DEVICE, do_cfg=True)
assert _cond_infer.shape == (4, 78, CFG.sd_cross_attn_dim), (
    f'Inference cond {_cond_infer.shape} != (4,78,{CFG.sd_cross_attn_dim})'
)
_uncond_half = _cond_infer[:2]
_cond_half   = _cond_infer[2:]
# Unconditional VLM token (last token in seq) should be zeros
assert _uncond_half[:, -1, :].abs().max().item() < 1e-6, (
    'Unconditional VLM token is not zero -- check null_vlm in build_combined_conditioning'
)
print(f'  ok  inference cond:  {tuple(_cond_infer.shape)} (CFG: uncond||cond)')
print(f'  ok  uncond VLM token zeros: {_uncond_half[:,-1,:].abs().max().item():.2e}')
print(f'\nbuild_combined_conditioning verified (byte-identical to Phase 2 §16).')


### §2.7 — Inference Functions (Phase 3 v3.0)

In [ ]:
# -- §25.1  run_vlm_inference + parse_vlm_output --------------------------------
#
# v3.0 note: the VLM still produces {edit_type, bbox, edit_description}.
# In v3.0 the bbox is treated as a *secondary* signal — preferred only when
# GroundingDINO returns no detections. The subject phrase is extracted from
# edit_description in §25.4 (extract_subject_phrase).

import torch, json as _json_mod, re as _re
from typing import Optional


def parse_vlm_output(generated_text: str) -> Optional[dict]:
    """Parse VLM generated text to extract {edit_type, bbox, edit_description}.

    Args:
        generated_text: raw string from model.generate() + batch_decode()

    Returns:
        dict with keys: edit_type (str), bbox (list[4 int]), edit_description (str)
        or None if parse fails.
    """
    text = generated_text.strip()
    _fence_match = _re.match(r"^```(?:json)?\s*\n?(.*?)\n?```$", text, _re.DOTALL)
    if _fence_match:
        text = _fence_match.group(1).strip()

    _json_match = _re.search(r"\{[^{}]*\}", text, _re.DOTALL)
    if _json_match and _json_match.group(0) != text:
        log.debug(f"parse_vlm_output: extracted JSON object from longer text")
        text = _json_match.group(0)

    try:
        data = _json_mod.loads(text)
    except _json_mod.JSONDecodeError as _e:
        log.warning(f"parse_vlm_output: JSON decode failed: {_e}\nRaw: {generated_text[:300]!r}")
        return None

    if "edit_type" not in data:
        log.warning(f"parse_vlm_output: missing edit_type. Raw: {generated_text[:200]!r}")
        return None
    if "bbox" not in data or not isinstance(data["bbox"], (list, tuple)) or len(data["bbox"]) != 4:
        log.warning(f"parse_vlm_output: invalid bbox field. Raw: {generated_text[:200]!r}")
        return None
    if "edit_description" not in data:
        log.warning(f"parse_vlm_output: missing edit_description. Raw: {generated_text[:200]!r}")
        return None

    try:
        bbox = [int(round(float(v))) for v in data["bbox"]]
    except (TypeError, ValueError) as _e:
        log.warning(f"parse_vlm_output: bbox value conversion failed: {_e}")
        return None

    if any(v < 0 or v > 1000 for v in bbox):
        log.warning(f"parse_vlm_output: bbox {bbox} outside [0,1000]; clipping.")
        bbox = [max(0, min(v, 1000)) for v in bbox]

    x1, y1, x2, y2 = bbox
    if x2 <= x1 or y2 <= y1:
        log.warning(
            f"parse_vlm_output: degenerate bbox {bbox} (zero/negative area). "
            f"Returning None — caller should use fallback."
        )
        return None

    edit_type = str(data["edit_type"]).strip().lower()
    if edit_type not in KNOWN_EDIT_TYPES:
        log.warning(
            f"parse_vlm_output: unknown edit_type {edit_type!r}; "
            f"known: {sorted(KNOWN_EDIT_TYPES)}. Keeping value."
        )

    return {
        "edit_type":        edit_type,
        "bbox":             bbox,
        "edit_description": str(data["edit_description"]).strip(),
    }


def run_vlm_inference(
    source_image,
    instruction: str,
    device: str = None,
) -> dict:
    """Run the fine-tuned VLM to predict edit_type, bbox, and edit_description.

    Returns a dict that always contains a usable bbox (falling back to
    CFG.phase3_fallback_bbox if parsing failed). The bbox here is the *VLM*
    bbox; v3.0 §26.1 will combine it with the grounding-model bbox.
    """
    _device = device or DEVICE

    if isinstance(source_image, (str, Path)):
        source_image = Image.open(source_image).convert("RGB")

    messages = build_messages(source_image, instruction, annotations=None)

    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(
        text           = [text_prompt],
        images         = image_inputs if image_inputs else None,
        videos         = video_inputs if video_inputs else None,
        padding        = False,
        return_tensors = "pt",
    )

    _input_ids  = inputs["input_ids"].to(_device)
    _attn_mask  = inputs["attention_mask"].to(_device)
    _extra_kw   = {}
    if "pixel_values" in inputs:
        _extra_kw["pixel_values"]   = inputs["pixel_values"].to(_device)
    if "image_grid_thw" in inputs:
        _extra_kw["image_grid_thw"] = inputs["image_grid_thw"].to(_device)

    with torch.no_grad():
        output_ids = vlm_model.generate(
            input_ids      = _input_ids,
            attention_mask = _attn_mask,
            max_new_tokens = CFG.phase3_max_new_tokens,
            do_sample      = False,
            pad_token_id   = vlm_processor.tokenizer.eos_token_id,
            **_extra_kw,
        )

    generated_ids = output_ids[:, _input_ids.shape[1]:]
    raw_text = vlm_processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0]

    parsed = parse_vlm_output(raw_text)

    if parsed is not None:
        return {
            "raw_text":         raw_text,
            "parsed":           parsed,
            "edit_type":        parsed["edit_type"],
            "bbox":             parsed["bbox"],
            "edit_description": parsed["edit_description"],
            "used_fallback":    False,
        }
    log.warning(
        f"run_vlm_inference: parse failed for instruction {instruction[:60]!r}. "
        f"Using fallback bbox {list(CFG.phase3_fallback_bbox)}."
    )
    return {
        "raw_text":         raw_text,
        "parsed":           None,
        "edit_type":        "unknown",
        "bbox":             list(CFG.phase3_fallback_bbox),
        "edit_description": instruction,
        "used_fallback":    True,
    }


# -- parse_vlm_output smoke tests -----------------------------------------
_good_json = '{"edit_type": "remove", "bbox": [100, 200, 600, 800], "edit_description": "remove the car"}'
_p = parse_vlm_output(_good_json)
assert _p is not None and _p["bbox"] == [100, 200, 600, 800]

_fenced = '```json\n{"edit_type": "add", "bbox": [0, 0, 500, 500], "edit_description": "add flowers"}\n```'
_p2 = parse_vlm_output(_fenced)
assert _p2 is not None and _p2["edit_type"] == "add"

_bad = "Sorry, I cannot do that."
_p3 = parse_vlm_output(_bad)
assert _p3 is None

_oob = '{"edit_type": "adjust", "bbox": [-10, 0, 1100, 500], "edit_description": "brighten"}'
_p4 = parse_vlm_output(_oob)
assert _p4 is not None and _p4["bbox"] == [0, 0, 1000, 500], f"OOB clip failed: {_p4}"

print("§25.1 run_vlm_inference + parse_vlm_output defined (v3.0).")
print("  ok  parse_vlm_output: valid JSON, fenced JSON, bad text, OOB bbox clipping")


In [ ]:
# -- §25.2  extract_hidden_state_inference -------------------------------------
#
# SPEC REQUIREMENT: 'Qwen hidden-state extraction must exclude image tokens and
# special/template tokens before mean pooling.'
#
# This function is IDENTICAL in logic to Phase 1 §11.1 extract_hidden_state_for_sample,
# but simplified for inference (no sample_meta dict -- takes image + instruction directly).
#
# Key invariants (must match Phase 1 §11.1):
#   - Same build_messages() call (user-turn only, add_generation_prompt=True)
#   - Same QWEN_SPECIAL_START threshold (151643)
#   - Same mean-pool over text tokens from last hidden layer (index -1)
#   - Same output dtype: float32, shape (vlm_hidden_dim,) on CPU
#
# Called once per image, before the DDIM denoising loop.

import torch
from typing import Optional


def extract_hidden_state_inference(
    source_image,
    instruction: str,
    device: str = None,
) -> Optional[torch.Tensor]:
    '''Extract VLM hidden state for SD conditioning (inference-time version).

    Runs a user-turn-only forward pass with output_hidden_states=True.
    Filters to text tokens (ID < QWEN_SPECIAL_START=151643) then mean-pools.
    Output is the same (hidden_dim,) float32 vector used in Phase 2 training,
    but extracted WITHOUT ground-truth annotation regions in the prompt
    (known distribution gap -- see Phase 3 notebook header for discussion).

    Args:
        source_image: PIL.Image.Image or Path/str
        instruction:  edit instruction string
        device:       target device (default DEVICE)

    Returns:
        Tensor (vlm_hidden_dim,) float32 on CPU, or None on failure.
        Returns None (not raises) -- caller should skip sample on failure.
    '''
    _device = device or DEVICE

    try:
        if isinstance(source_image, (str, Path)):
            source_image = Image.open(source_image).convert('RGB')

        # Build user-turn messages (no annotations -- matches Phase 3 generation prompt)
        messages = build_messages(source_image, instruction, annotations=None)

        # Apply chat template (user-turn only, with generation prompt)
        text_prompt = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = vlm_processor(
            text           = [text_prompt],
            images         = image_inputs if image_inputs else None,
            videos         = video_inputs if video_inputs else None,
            padding        = False,
            return_tensors = 'pt',
        )

        input_ids = inputs['input_ids'].to(_device)
        attn_mask = inputs['attention_mask'].to(_device)
        extra = {}
        if 'pixel_values' in inputs:
            extra['pixel_values']   = inputs['pixel_values'].to(_device)
        if 'image_grid_thw' in inputs:
            extra['image_grid_thw'] = inputs['image_grid_thw'].to(_device)

        with torch.no_grad():
            outputs = vlm_model(
                input_ids            = input_ids,
                attention_mask       = attn_mask,
                output_hidden_states = True,
                **extra,
            )

        # Last hidden layer: (1, seq_len, hidden_dim)
        last_hidden = outputs.hidden_states[-1]
        assert last_hidden.ndim == 3 and last_hidden.shape[0] == 1, (
            f'Unexpected last_hidden shape: {last_hidden.shape}'
        )

        id_list = input_ids[0].tolist()
        seq_len = last_hidden.shape[1]
        assert len(id_list) == seq_len, (
            f'id_list len {len(id_list)} != seq_len {seq_len}'
        )

        # Boolean mask: True = text token (keep for mean pool), False = special/image (exclude)
        text_mask = torch.tensor(
            [tok_id < QWEN_SPECIAL_START for tok_id in id_list],
            dtype=torch.bool, device=_device,
        )  # (seq_len,)

        n_text_tokens = text_mask.sum().item()
        if n_text_tokens == 0:
            log.warning(
                f'extract_hidden_state_inference: no text tokens found '
                f'(all {seq_len} token IDs >= QWEN_SPECIAL_START={QWEN_SPECIAL_START}). '
                f'Returning None.'
            )
            return None

        # Mean pool text tokens: (n_text, hidden_dim) -> (hidden_dim,)
        text_hidden = last_hidden[0][text_mask]  # (n_text, hidden_dim)
        pooled      = text_hidden.mean(dim=0)    # (hidden_dim,)

        assert pooled.shape == (CFG.vlm_hidden_dim,), (
            f'Pooled shape {pooled.shape} != ({CFG.vlm_hidden_dim},). '
            f'Check CFG.vlm_hidden_dim matches the loaded model.'
        )

        log.debug(
            f'extract_hidden_state_inference: seq_len={seq_len}, '
            f'text_tokens={n_text_tokens} '
            f'({100*n_text_tokens/seq_len:.1f}%), '
            f'range=[{pooled.min().item():.3f}, {pooled.max().item():.3f}]'
        )

        return pooled.float().cpu()   # float32 on CPU

    except Exception as _e:
        log.warning(f'extract_hidden_state_inference failed: {type(_e).__name__}: {_e}')
        return None


print('§25.2 extract_hidden_state_inference() defined.')
print(f'  QWEN_SPECIAL_START = {QWEN_SPECIAL_START}  (text tokens: ID < threshold)')
print(f'  Output: ({CFG.vlm_hidden_dim},) float32 CPU')
print(f'  Forward: user-turn only (add_generation_prompt=True, no assistant tokens)')
print(f'  Pooling: mean over text tokens, last hidden layer')


In [ ]:
# -- §25.4  extract_subject_phrase  (v3.0) -----------------------------------
#
# Goal: produce a short noun phrase suitable for GroundingDINO from
# (a) the VLM's edit_description and (b) the original NL instruction.
#
# Strategy (rule-based — deliberate, no extra model required):
#   1. Strip common edit-action verbs/prefixes that won't help grounding
#      ("turn the X into ...", "change X to ...", "replace X with ...",
#       "remove X", "add X", "make the X ...").
#   2. After stripping, take the FIRST 1–4 word noun-phrase chunk.
#   3. Drop trailing modifiers introduced by " into "/" to "/" with " — these
#      describe the *target* state (red, white crust), not what to localize.
#   4. Fall back to the noun head extracted from the original instruction if
#      the description is empty or all-verb.
#
# This is intentionally not perfect — GroundingDINO is forgiving about extra
# words ("motorcycle on the asphalt" still detects motorcycles), so we err on
# the side of keeping more content rather than over-pruning. If the rule
# returns garbage the §26.1 pipeline still falls back to the VLM bbox.

import re as _re_subj

# Verbs/phrases that reliably precede the *subject* in ImgEdit-style instructions.
# Order matters: we strip from the longest pattern first.
_LEADING_PATTERNS = [
    r"^turn\s+(?:the\s+|a\s+|an\s+)?",
    r"^change\s+(?:the\s+|a\s+|an\s+)?",
    r"^replace\s+(?:the\s+|a\s+|an\s+)?",
    r"^remove\s+(?:the\s+|a\s+|an\s+)?",
    r"^delete\s+(?:the\s+|a\s+|an\s+)?",
    r"^add\s+(?:a\s+|an\s+|the\s+)?",
    r"^make\s+(?:the\s+|a\s+|an\s+)?",
    r"^edit\s+(?:the\s+|a\s+|an\s+)?",
    r"^modify\s+(?:the\s+|a\s+|an\s+)?",
    r"^paint\s+(?:the\s+|a\s+|an\s+)?",
    r"^recolour\s+(?:the\s+|a\s+|an\s+)?",
    r"^recolor\s+(?:the\s+|a\s+|an\s+)?",
    r"^transform\s+(?:the\s+|a\s+|an\s+)?",
]

# Trailing-modifier markers — anything to their RIGHT describes the target
# state, not the object to localize.
_TARGET_MARKERS = (
    " into ", " to a ", " to an ", " to the ", " with ", " using ",
    " so that ", " so it ", " in a ", " in the style of ",
)

# Localization-noise tokens that often appear in VLM descriptions and that
# we should drop when they are the *trailing* word (positional fluff).
_TRAILING_NOISE = {
    "area", "region", "section", "part", "side",
    "image", "picture", "scene", "frame", "centre", "center",
    "background", "foreground",
}

# Filler positional phrases — strip these mid-string. Order matters.
_POSITIONAL_FRAGMENTS = [
    "positioned in the upper-right area",
    "positioned in the upper-left area",
    "positioned in the lower-right area",
    "positioned in the lower-left area",
    "positioned in the central area",
    "positioned in the upper area",
    "positioned in the lower area",
    "positioned in the right area",
    "positioned in the left area",
    "positioned in the centre area",
    "positioned in the center area",
    "positioned at the upper right",
    "positioned at the upper left",
    "positioned at the lower right",
    "positioned at the lower left",
    "in the upper-right area",
    "in the upper-left area",
    "in the lower-right area",
    "in the lower-left area",
    "in the central area",
    "in the centre",
    "in the center",
]


def extract_subject_phrase(
    edit_description: str,
    instruction: str = "",
) -> str:
    """Extract a short noun phrase suitable as a GroundingDINO prompt.

    Args:
        edit_description: from the VLM's JSON output.
        instruction: original NL instruction (used as fallback).

    Returns:
        A lowercased noun phrase, or "" if extraction failed.
    """
    candidates = [s for s in (edit_description or "", instruction or "") if s.strip()]
    for raw in candidates:
        text = raw.strip().lower()

        # 1. Strip filler positional fragments first (so they don't interfere
        #    with leading-pattern matching).
        for frag in _POSITIONAL_FRAGMENTS:
            text = text.replace(frag, " ")
        text = _re_subj.sub(r"\s+", " ", text).strip()

        # 2. Strip leading edit-action prefix.
        for pat in _LEADING_PATTERNS:
            new = _re_subj.sub(pat, "", text)
            if new != text:
                text = new.strip()
                break

        # 3. Truncate at the first target marker.
        lower = text.lower()
        cut = len(lower)
        for marker in _TARGET_MARKERS:
            i = lower.find(marker)
            if i >= 0 and i < cut:
                cut = i
        text = text[:cut].strip()

        # 4. Drop trailing punctuation.
        text = text.rstrip(".,;:!?- ").strip()

        # 5. Drop trailing positional-noise words.
        toks = text.split()
        while toks and toks[-1] in _TRAILING_NOISE:
            toks.pop()
        text = " ".join(toks)

        # 6. Cap to first 5 tokens — GroundingDINO loses precision on very
        #    long noun-phrase prompts.
        toks = text.split()
        if len(toks) > 5:
            toks = toks[:5]
        text = " ".join(toks).strip()

        if text:
            return text

    return ""


# -- Smoke tests -----------------------------------------------------------
_t = extract_subject_phrase("Turn motorcycle positioned in the central area into red")
assert _t == "motorcycle", f"motorcycle test failed: {_t!r}"

_t = extract_subject_phrase("Turn bread positioned in the upper-right area into white crust")
assert _t == "bread", f"bread test failed: {_t!r}"

_t = extract_subject_phrase("Turn text positioned in the central area into red festive font")
assert _t == "text", f"text test failed: {_t!r}"

_t = extract_subject_phrase("Replace the dog with a cat")
assert _t == "dog", f"replace-dog test failed: {_t!r}"

_t = extract_subject_phrase("Change shirt colour to blue")
assert _t.startswith("shirt"), f"shirt test failed: {_t!r}"

_t = extract_subject_phrase("Remove the car from the parking lot")
assert _t.startswith("car"), f"car test failed: {_t!r}"

_t = extract_subject_phrase("", "remove the bicycle")
assert _t.startswith("bicycle"), f"fallback-instruction test failed: {_t!r}"

_t = extract_subject_phrase("")
assert _t == "", f"empty test failed: {_t!r}"

print("§25.4 extract_subject_phrase defined (v3.0).")
print("  ok  motorcycle / bread / text / dog / shirt / car / fallback / empty")


In [ ]:
# -- §26.1  bbox_to_sam2_mask  (v3.0 — grounding-model + multi-mask + rerank)
#
# Pipeline (per call):
#   1.  Build a subject phrase via §25.4 extract_subject_phrase().
#   2.  Run GroundingDINO with that phrase. Returns N candidate (bbox, score).
#   3.  Sanity-check the VLM bbox (degenerate / out-of-bounds detection).
#   4.  Pick the bbox to feed SAM2:
#         - GroundingDINO top-score box if available.
#         - Else fall back to VLM bbox (provided it is non-degenerate).
#         - Else fall back to the global-edit short-circuit (full-image mask).
#   5.  Run SAM2 with multimask_output=True + (box, optional centre point).
#       Returns up to 3 candidate masks.
#   6.  Rerank candidates by CLIP image-text similarity to the subject phrase.
#       (When CLIP rerank is disabled the highest-score SAM2 mask is used.)
#   7.  Mask post-processing: morphological closing (kernel=CFG.mask_morph_close_kernel)
#       and largest-connected-component selection.
#
# All decisions and per-step diagnostics are returned via the `diagnostics`
# dict alongside the mask, so the batch-inference loop can persist them for
# later analysis.

import numpy as np
import torch
import torch.nn.functional as _F
from PIL import Image
from typing import Optional, Tuple, List
import scipy.ndimage as _scipy_ndimage  # required by _mask_postprocess


def _bbox_xyxy_to_centre_point(bbox_abs: List[int]) -> Tuple[int, int]:
    x1, y1, x2, y2 = bbox_abs
    return (int(round((x1 + x2) / 2)), int(round((y1 + y2) / 2)))


def _bbox_area_frac(bbox_abs: List[int], W: int, H: int) -> float:
    x1, y1, x2, y2 = bbox_abs
    return max(0.0, (x2 - x1) * (y2 - y1)) / max(1.0, float(W * H))


def _is_vlm_bbox_degenerate(bbox_rel: List[int]) -> bool:
    """Detect VLM bboxes that we should not trust spatially.

    Returns True for:
      - Full-image / near-full-image boxes (>= bbox_degenerate_max_frac).
      - Pinpoint / tiny boxes (<= bbox_degenerate_min_frac).
      - Boxes flagged by is_full_image_bbox (the [0,0,1000,1000] fallback).
    """
    if is_full_image_bbox(bbox_rel):
        return True
    x1, y1, x2, y2 = bbox_rel
    rel_area = max(0, x2 - x1) * max(0, y2 - y1) / 1_000_000.0
    if rel_area >= CFG.bbox_degenerate_max_frac:
        return True
    if rel_area <= CFG.bbox_degenerate_min_frac:
        return True
    return False


def _mask_postprocess(mask_uint8: np.ndarray) -> np.ndarray:
    """Morphological closing + largest-connected-component selection.

    Args:
        mask_uint8: (H, W) uint8 mask, 0 = bg, 255 = fg.

    Returns:
        Cleaned (H, W) uint8 mask with the same polarity.
    """
    H, W = mask_uint8.shape
    bin_mask = mask_uint8 > 127

    # 1. Morphological closing fills small holes / thin gaps.
    if CFG.mask_morph_close_kernel and CFG.mask_morph_close_kernel > 1:
        k = int(CFG.mask_morph_close_kernel)
        bin_mask = _scipy_ndimage.binary_closing(
            bin_mask, structure=np.ones((k, k), dtype=bool)
        )

    # 2. Largest connected component (drop speckle).
    if CFG.mask_keep_largest_component:
        labels, n_cc = _scipy_ndimage.label(bin_mask)
        if n_cc > 1:
            sizes = _scipy_ndimage.sum(bin_mask, labels, range(1, n_cc + 1))
            largest = int(np.argmax(sizes)) + 1
            bin_mask = labels == largest
        elif n_cc == 0:
            return np.zeros_like(mask_uint8)

    # 3. Drop the whole mask if it is below the minimum area fraction.
    if bin_mask.mean() < CFG.mask_min_component_area_frac:
        log.warning(
            "mask_postprocess: mask area %.4f < min %.4f — returning original mask",
            bin_mask.mean(), CFG.mask_min_component_area_frac,
        )
        return mask_uint8

    return (bin_mask.astype(np.uint8) * 255)


def _clip_rerank_masks(
    image_pil: Image.Image,
    masks: List[np.ndarray],
    text_phrase: str,
) -> int:
    """Pick the index of the mask whose masked image best matches `text_phrase`
    in CLIP image-text similarity space.

    We reuse the CLIPTextModel + tokenizer that SD already loaded for text
    conditioning. Image features come from the SD VAE-paired CLIP-vision encoder
    that we DON'T have on hand, so we use a lightweight proxy: average pixel
    embedding via the SD CLIP text branch is not appropriate. Instead we use
    transformers.CLIPModel (vit-base-patch32) which we lazily load once and cache.
    """
    global _clip_rerank_model, _clip_rerank_processor
    try:
        _ = _clip_rerank_model
    except NameError:
        from transformers import CLIPModel, CLIPProcessor
        _CLIP_RERANK_ID = "openai/clip-vit-base-patch32"
        log.info(f"_clip_rerank: lazy-loading {_CLIP_RERANK_ID} for mask rerank")
        _clip_rerank_processor = CLIPProcessor.from_pretrained(_CLIP_RERANK_ID)
        _clip_rerank_model = CLIPModel.from_pretrained(_CLIP_RERANK_ID).to(DEVICE).eval()
        for _p in _clip_rerank_model.parameters():
            _p.requires_grad_(False)

    arr = np.array(image_pil.convert("RGB"))
    masked_imgs = []
    for m in masks:
        # Build a 3-channel masked image: mask region kept, background blacked.
        # CLIP's training distribution doesn't see hard cuts cleanly, so we also
        # tightly crop to the mask bbox to give CLIP a natural-looking patch.
        ys, xs = np.where(m > 127)
        if ys.size == 0 or xs.size == 0:
            # Empty mask — give CLIP the whole image so it still gets *some* score.
            masked_imgs.append(image_pil)
            continue
        y1, y2 = int(ys.min()), int(ys.max()) + 1
        x1, x2 = int(xs.min()), int(xs.max()) + 1
        crop = arr[y1:y2, x1:x2]
        masked_imgs.append(Image.fromarray(crop))

    inputs = _clip_rerank_processor(
        text=[text_phrase] * len(masked_imgs),
        images=masked_imgs,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(DEVICE)

    with torch.no_grad():
        out = _clip_rerank_model(**inputs)
    sims = out.logits_per_image.diag().cpu().numpy()  # one per (image, text) pair
    best = int(np.argmax(sims))
    log.info(f"_clip_rerank: similarities={sims.tolist()} -> picked {best}")
    return best


def bbox_to_sam2_mask(
    source_image: Image.Image,
    bbox_rel: list,
    edit_description: str = "",
    instruction: str = "",
    device=None,
) -> Tuple[np.ndarray, dict]:
    """v3.0 mask generation: grounding-model + SAM2 multi-mask + rerank.

    Args:
        source_image: PIL Image (any mode, will be converted to RGB).
        bbox_rel: VLM-predicted [x1,y1,x2,y2] in [0,1000] relative coords.
        edit_description: from VLM JSON output (used to derive subject phrase).
        instruction: original NL edit instruction (fallback for subject extraction).
        device: torch device (kept for API symmetry; not used here).

    Returns:
        (mask_uint8, diagnostics)
            mask_uint8: (H, W) uint8 mask with 0=preserve, 255=inpaint.
            diagnostics: dict with per-step info for the batch loop.
    """
    assert sam2_predictor is not None, "sam2_predictor not loaded — run §21.2"
    assert grounding_model is not None, "grounding model not loaded — run §22.5"

    diag = {
        "vlm_bbox_rel": list(bbox_rel),
        "vlm_bbox_degenerate": False,
        "subject_phrase": "",
        "grounding_n_boxes": 0,
        "grounding_top_score": None,
        "grounding_top_bbox_abs": None,
        "selected_source": None,            # 'grounding' | 'vlm' | 'global_edit'
        "selected_bbox_abs": None,
        "sam2_n_masks": 0,
        "sam2_scores": [],
        "rerank_used": False,
        "rerank_picked": None,
        "post_clean_kept_largest": CFG.mask_keep_largest_component,
        "final_mask_coverage": None,
    }

    img_rgb = source_image.convert("RGB")
    W, H = img_rgb.size
    img_area = W * H

    # ── Step 1: subject phrase ───────────────────────────────────────────
    subject_phrase = extract_subject_phrase(
        edit_description=edit_description, instruction=instruction
    )
    diag["subject_phrase"] = subject_phrase

    # ── Step 2: GroundingDINO detection ──────────────────────────────────
    grounding_boxes_abs = []   # list of (bbox_xyxy_abs, score)
    if subject_phrase:
        grounding_boxes_abs = run_grounding_dino(
            image=img_rgb, phrase=subject_phrase
        )
        diag["grounding_n_boxes"] = len(grounding_boxes_abs)
        if grounding_boxes_abs:
            top_box, top_score = grounding_boxes_abs[0]
            diag["grounding_top_score"] = float(top_score)
            diag["grounding_top_bbox_abs"] = list(map(int, top_box))

    # ── Step 3: VLM-bbox sanity check ────────────────────────────────────
    diag["vlm_bbox_degenerate"] = _is_vlm_bbox_degenerate(bbox_rel)

    # ── Step 4: select bbox to drive SAM2 ─────────────────────────────────
    selected_bbox_abs = None
    selected_source = None

    if grounding_boxes_abs:
        selected_bbox_abs = list(map(int, grounding_boxes_abs[0][0]))
        selected_source = "grounding"
    elif not diag["vlm_bbox_degenerate"]:
        x1_a, y1_a, x2_a, y2_a = bbox_relative_to_absolute(bbox_rel, W, H)
        selected_bbox_abs = [x1_a, y1_a, x2_a, y2_a]
        selected_source = "vlm"
    else:
        # No reliable spatial signal — fall through to global-edit branch below.
        selected_source = "global_edit"

    # Global-edit short-circuit (covers both the explicit case and the fallback
    # path above). Returns a uniform full-image mask without invoking SAM2.
    if (
        selected_source == "global_edit"
        or (
            selected_bbox_abs is not None
            and _bbox_area_frac(selected_bbox_abs, W, H) >= CFG.global_edit_area_frac
        )
    ):
        log.info(
            "bbox_to_sam2_mask: global-edit short-circuit fired (source=%s) — "
            "returning full-image mask.", selected_source,
        )
        diag["selected_source"] = "global_edit"
        diag["selected_bbox_abs"] = [0, 0, W, H]
        full_mask = np.full((H, W), 255, dtype=np.uint8)
        diag["final_mask_coverage"] = 1.0
        return full_mask, diag

    # Clamp the chosen bbox to image bounds.
    x1_a, y1_a, x2_a, y2_a = selected_bbox_abs
    x1_a = max(0, min(x1_a, W - 1))
    y1_a = max(0, min(y1_a, H - 1))
    x2_a = max(x1_a + 1, min(x2_a, W))
    y2_a = max(y1_a + 1, min(y2_a, H))
    selected_bbox_abs = [x1_a, y1_a, x2_a, y2_a]
    diag["selected_source"] = selected_source
    diag["selected_bbox_abs"] = selected_bbox_abs

    # ── Step 5: SAM2 multi-mask ──────────────────────────────────────────
    box_np = np.array([selected_bbox_abs], dtype=np.float32)
    point_coords = None
    point_labels = None
    if CFG.sam2_use_centre_point:
        cx, cy = _bbox_xyxy_to_centre_point(selected_bbox_abs)
        point_coords = np.array([[cx, cy]], dtype=np.float32)
        point_labels = np.array([1], dtype=np.int32)   # 1 = positive

    sam2_predictor.set_image(np.array(img_rgb))
    masks, scores, _logits = sam2_predictor.predict(
        point_coords=point_coords,
        point_labels=point_labels,
        box=box_np,
        multimask_output=CFG.sam2_multimask_output,
    )
    # masks shape:
    #   multimask=True  -> (3, H, W) bool
    #   multimask=False -> (1, H, W) bool
    diag["sam2_n_masks"] = int(masks.shape[0])
    diag["sam2_scores"]  = [float(s) for s in scores.tolist()]

    masks_uint8 = [(m.astype(np.uint8) * 255) for m in masks]

    # ── Step 6: rerank ────────────────────────────────────────────────────
    if (
        CFG.sam2_clip_rerank
        and len(masks_uint8) > 1
        and subject_phrase
    ):
        try:
            best_idx = _clip_rerank_masks(img_rgb, masks_uint8, subject_phrase)
            diag["rerank_used"]  = True
            diag["rerank_picked"] = int(best_idx)
        except Exception as _e:
            log.warning(f"_clip_rerank_masks failed: {_e!r} — falling back to top SAM2 score")
            best_idx = int(np.argmax(scores))
    else:
        best_idx = int(np.argmax(scores))
    chosen_mask = masks_uint8[best_idx]

    # ── Step 7: post-process ─────────────────────────────────────────────
    cleaned = _mask_postprocess(chosen_mask)
    cov = float((cleaned > 127).mean())
    diag["final_mask_coverage"] = cov

    log.info(
        "bbox_to_sam2_mask v3: source=%s subject=%r grounding_n=%d sam_scores=%s "
        "picked=%d coverage=%.3f",
        diag["selected_source"], subject_phrase, diag["grounding_n_boxes"],
        diag["sam2_scores"], best_idx, cov,
    )

    assert cleaned.shape == (H, W), (
        f"Mask shape {cleaned.shape} != image shape ({H},{W})"
    )
    return cleaned, diag


print("bbox_to_sam2_mask defined (v3.0 — grounding + multi-mask + rerank).")


In [ ]:
# -- §27.1  run_inpainting --------------------------------------------------
#
# SPEC REQUIREMENT: '9-channel UNet input: noisy_target(4) | mask(1) | masked_source(4)'
# SPEC REQUIREMENT: 'Guidance behavior must be handled correctly when passing prompt_embeds'
# SPEC REQUIREMENT: 'DDIM scheduler, 20 inference steps'
#
# Mask polarity (critical -- easy to get wrong):
#   mask = 1 where we INPAINT (replace/edit region)
#   mask = 0 where we PRESERVE (keep source pixels)
#   SAM2 output: 255 in the target region -> normalise to [0,1].
#   masked_source_latent encodes the source image with the edit region ZEROED OUT.
#
# CFG shape guard:
#   build_combined_conditioning(do_cfg=True) returns (2, 78, 768).
#   We tile the latent/mask inputs to batch=2 to match.
#   After UNet forward: split noise_pred into [uncond, cond] along dim=0.
#   CFG formula: noise = uncond + guidance_scale * (cond - uncond)

import torch
import numpy as np
from PIL import Image
from typing import Optional

def run_inpainting(
    source_image: Image.Image,
    vlm_hidden: torch.Tensor,
    instruction: str,
    mask_np: np.ndarray,
    device=None,
    seed: Optional[int] = None,
) -> Image.Image:
    """Run SD 1.5 inpainting with CFG on source_image in the mask region.

    Args:
        source_image: PIL Image (RGB).
        vlm_hidden:   float32 Tensor of shape (vlm_hidden_dim,) on CPU.
                      From extract_hidden_state_inference().
        instruction:  text instruction for CLIP conditioning.
        mask_np:      uint8 ndarray (H,W) with 255=edit region, 0=preserve.
        device:       torch.device. Defaults to DEVICE.
        seed:         Optional int for reproducible inference.

    Returns:
        Edited PIL Image (RGB, same size as source_image).
    """
    if device is None:
        device = DEVICE

    assert vae is not None and unet is not None, "Load SD components first (§23)"
    assert vlm_adapter is not None, "Load VLMProjectionAdapter first (§23.2)"

    H_orig, W_orig = source_image.size[1], source_image.size[0]
    target_size = CFG.phase3_resolution  # 512

    # -- 1. Resize image and mask to 512x512 --------------------------------
    img_resized = source_image.convert('RGB').resize(
        (target_size, target_size), Image.LANCZOS
    )
    # Resize mask: nearest-neighbour to preserve binary values.
    mask_pil = Image.fromarray(mask_np).resize(
        (target_size, target_size), Image.NEAREST
    )
    mask_512 = np.array(mask_pil)

    # -- 2. Build float mask tensor (1,1,H,W) --------------------------------
    # 1 = inpaint, 0 = preserve. SAM2 uses 255=foreground -> divide by 255.
    mask_t = torch.from_numpy(mask_512.astype(np.float32) / 255.0)
    mask_t = mask_t.unsqueeze(0).unsqueeze(0).to(device)  # (1,1,H,W)

    # -- 3. VAE encode source image (fp32) -----------------------------------
    img_arr = np.array(img_resized).astype(np.float32) / 127.5 - 1.0
    img_t = torch.from_numpy(img_arr).permute(2, 0, 1).unsqueeze(0).to(device)  # (1,3,H,W)

    vae.to(device)
    with torch.no_grad():
        # encode in fp32; scale by VAE_SCALE_FACTOR (0.18215)
        vae_fp32 = vae.float()
        source_latent = vae_fp32.encode(img_t).latent_dist.sample()
        source_latent = source_latent * CFG.vae_scale_factor  # (1,4,H/8,W/8)

    latent_H, latent_W = source_latent.shape[2], source_latent.shape[3]
    # shape guard
    assert source_latent.shape == (1, 4, latent_H, latent_W), (
        f"source_latent shape {source_latent.shape} unexpected"
    )

    # -- 4. Build masked source latent ---------------------------------------
    # Resize mask to latent spatial dims (H/8, W/8) for masking source latent.
    mask_latent_np = (mask_512.astype(np.float32) / 255.0)
    mask_latent_pil = Image.fromarray((mask_latent_np * 255).astype(np.uint8)).resize(
        (latent_W, latent_H), Image.NEAREST
    )
    mask_for_latent = torch.from_numpy(
        np.array(mask_latent_pil).astype(np.float32) / 255.0
    ).to(device).unsqueeze(0).unsqueeze(0)  # (1,1,latent_H,latent_W)

    # Zero out source latent in the edit region -> masked_source_latent
    masked_source_latent = source_latent * (1.0 - mask_for_latent)  # (1,4,Hl,Wl)

    # UNet-sized mask: (1,1,latent_H,latent_W)
    mask_for_unet = mask_for_latent  # already at latent spatial size

    # -- 5. Initialize noisy latent ------------------------------------------
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)
    else:
        generator = None

    noise_scheduler.set_timesteps(CFG.phase3_num_inference_steps)
    timesteps = noise_scheduler.timesteps

    init_noise = torch.randn(
        (1, 4, latent_H, latent_W),
        device=device, dtype=torch.float32,
        generator=generator,
    )
    # Scale initial noise by scheduler's sigma (DDIM convention).
    latent = init_noise * noise_scheduler.init_noise_sigma

    # -- 6. Build combined conditioning (CLIP + VLM projection) -------------
    # do_cfg=True -> returns (2*batch, 78, 768): rows 0..B-1 = uncond,
    # rows B..2B-1 = cond. With batch=1, that is (2, 78, 768).
    # extract_hidden_state_inference() returns shape (vlm_hidden_dim,) on CPU;
    # build_combined_conditioning expects (batch, vlm_hidden_dim), so we add
    # the batch dim via unsqueeze(0). The keyword is `vlm_hiddens` (plural) --
    # this must stay byte-identical to Phase 2 §16; do NOT rename the function.
    if vlm_hidden.dim() == 1:
        vlm_hiddens_b = vlm_hidden.unsqueeze(0)  # (1, vlm_hidden_dim)
    else:
        vlm_hiddens_b = vlm_hidden
    assert vlm_hiddens_b.shape == (1, CFG.vlm_hidden_dim), (
        f'vlm_hiddens batched shape {tuple(vlm_hiddens_b.shape)} != (1, {CFG.vlm_hidden_dim})'
    )
    encoder_hidden_states = build_combined_conditioning(
        instructions=[instruction],
        vlm_hiddens=vlm_hiddens_b,
        device=device,
        do_cfg=True,
    )  # (2, 78, 768)
    assert encoder_hidden_states.shape[0] == 2, (
        f"CFG conditioning must have batch=2, got {encoder_hidden_states.shape}"
    )
    assert encoder_hidden_states.shape[1] == 78, (
        f"Conditioning seq_len must be 78 (77 CLIP + 1 VLM), got shape {encoder_hidden_states.shape}"
    )

    # Cast conditioning to fp16 for UNet.
    encoder_hidden_states = encoder_hidden_states.to(dtype=torch.float16)

    # Cast SD components to fp16.
    unet.to(device=device, dtype=torch.float16)

    # -- 7. DDIM denoising loop ----------------------------------------------
    latent = latent.to(dtype=torch.float16)
    source_latent = source_latent.to(dtype=torch.float16)
    masked_source_latent = masked_source_latent.to(dtype=torch.float16)
    mask_for_unet = mask_for_unet.to(dtype=torch.float16)

    unet.eval()
    with torch.no_grad():
        for t in timesteps:
            # Tile latent and mask inputs for CFG batch=2.
            latent_model_input = torch.cat([latent, latent], dim=0)  # (2,4,Hl,Wl)
            mask_input = torch.cat([mask_for_unet, mask_for_unet], dim=0)  # (2,1,Hl,Wl)
            masked_src_input = torch.cat(
                [masked_source_latent, masked_source_latent], dim=0
            )  # (2,4,Hl,Wl)

            # Concatenate along channel dim: (2, 4+1+4, Hl, Wl) = (2,9,Hl,Wl)
            unet_input = torch.cat(
                [latent_model_input, mask_input, masked_src_input], dim=1
            )  # (2, 9, Hl, Wl)

            assert unet_input.shape[1] == 9, (
                f"UNet input must have 9 channels, got {unet_input.shape[1]}"
            )

            # Scale input for DDIM.
            unet_input = noise_scheduler.scale_model_input(unet_input, t)

            # UNet forward: predict noise.
            noise_pred = unet(
                unet_input,
                t,
                encoder_hidden_states=encoder_hidden_states,
            ).sample  # (2, 4, Hl, Wl)

            # Apply CFG: split uncond and cond predictions.
            noise_pred_uncond, noise_pred_cond = noise_pred.chunk(2, dim=0)
            noise_pred_cfg = noise_pred_uncond + CFG.phase3_guidance_scale * (
                noise_pred_cond - noise_pred_uncond
            )

            # Cast to float32 for scheduler step (avoids fp16 overflow in scheduler).
            latent = noise_scheduler.step(
                noise_pred_cfg.float(), t, latent.float()
            ).prev_sample.to(dtype=torch.float16)

    # -- 8. VAE decode -------------------------------------------------------
    with torch.no_grad():
        # Unscale latent before decoding.
        latent_fp32 = latent.float() / CFG.vae_scale_factor
        decoded = vae_fp32.decode(latent_fp32).sample  # (1, 3, H, W) in [-1,1]

    # Convert to uint8 PIL image.
    decoded_np = decoded[0].permute(1, 2, 0).cpu().numpy()
    decoded_np = np.clip((decoded_np + 1.0) * 127.5, 0, 255).astype(np.uint8)
    # NOTE: resize is now handled inside the compositing block below.
    # (We keep decoded_np as 512x512 for compositing, then resize the composite.)

    # -- 9. Composite: paste generated pixels onto source for preserved regions --
    # WHY: The DDIM loop fully regenerates the entire latent from noise, so the
    # decoded image replaces all pixels -- including ones the mask says to preserve.
    # A well-trained model learns to reconstruct background faithfully, but during
    # early training (few epochs) it hallucinates the background, causing the
    # "full-image distortion" artifact. Hard-compositing the source pixels back into
    # the preserve region (mask=0) guarantees zero background drift regardless of
    # training maturity, at no quality cost in the edit region.
    #
    # PIL.Image.composite(foreground, background, mask):
    #   mask white (255) -> use foreground (generated)
    #   mask black (0)   -> use background (source)
    #
    # We use the 512-resized source image (img_resized) and the 512-resized mask
    # (mask_512) to composite at the working resolution, then resize the result
    # back to the original image dimensions.
    #
    # IMPORTANT: result_img is already resized to (W_orig, H_orig) by the resize
    # call above. We composite at 512 first (before that resize) and then resize,
    # to keep the compositing in latent-aligned pixel space and avoid double-
    # interpolation artefacts on mask edges.

    # Composite at 512x512 before the final resize.
    # decoded_np is already uint8 (H=512, W=512, C=3) at this point.
    generated_512 = Image.fromarray(decoded_np)  # (512, 512) RGB

    # Build a 1-channel composite mask from mask_512.
    # mask_512 is uint8 (H, W) with 255=inpaint, 0=preserve.
    composite_mask_512 = Image.fromarray(mask_512, mode='L')

    # composite(foreground=generated, background=source, mask)
    composited_512 = Image.composite(generated_512, img_resized, composite_mask_512)

    # Now resize the composited result to original image dimensions.
    result_img = composited_512.resize((W_orig, H_orig), Image.LANCZOS)

    return result_img



print("run_inpainting defined.")


In [ ]:
# -- §28.1  run_inference_pipeline  (v3.0 — grounding-model branch) ---------
#
# Step ordering UNCHANGED:
#   1. VLM forward generation -> {edit_type, bbox, edit_description}.
#   2. VLM hidden-state extraction -> 2048-d conditioning vector.
#   3. Mask generation (v3.0: grounding -> SAM2 multi-mask -> rerank -> post).
#   4. SD inpainting with CFG, byte-identical to v2.2.
#
# What's new at the pipeline level:
#   * Step 3 receives the edit_description so the grounding model can phrase-prompt.
#   * Diagnostics dict from §26.1 is propagated into the metadata block, so
#     §29.1 can persist per-sample localization decisions for evaluation.

import time
import torch
from PIL import Image


def run_inference_pipeline(
    source_image: Image.Image,
    instruction: str,
    seed: int = None,
    device=None,
) -> dict:
    """Full Phase 3 inference pipeline (v3.0): VLM -> grounding -> SAM2 -> SD.

    Returns:
        dict with keys:
          edited_image  -- PIL Image (same size as source_image)
          vlm_result    -- dict from run_vlm_inference()
          mask_np       -- uint8 ndarray (H,W)
          mask_diag     -- per-sample mask diagnostics from §26.1
          vlm_hidden    -- Tensor (vlm_hidden_dim,) float32 CPU, or None
          elapsed_s     -- total wall-clock seconds
          metadata      -- dict with per-step timing and diagnostics
    """
    if device is None:
        device = DEVICE

    meta = {}
    t0 = time.time()

    # ── Step 1: VLM inference (still used for edit_type, JSON, bbox-as-fallback) ─
    t1 = time.time()
    vlm_result = run_vlm_inference(
        source_image=source_image,
        instruction=instruction,
        device=device,
    )
    meta["vlm_inference_s"] = time.time() - t1

    bbox_rel        = vlm_result["bbox"]
    edit_type       = vlm_result["edit_type"]
    edit_description = vlm_result.get("edit_description", instruction)
    used_fallback   = vlm_result.get("used_fallback", False)

    log.info(
        "Pipeline step 1: edit_type=%s bbox=%s desc=%r fallback=%s (%.2fs)",
        edit_type, bbox_rel, edit_description[:60], used_fallback,
        meta["vlm_inference_s"],
    )

    # ── Step 2: VLM hidden state extraction (UNCHANGED — Phase 2 conditioning) ──
    t2 = time.time()
    vlm_hidden = extract_hidden_state_inference(
        source_image=source_image,
        instruction=instruction,
        device=device,
    )
    meta["hidden_state_s"] = time.time() - t2

    if vlm_hidden is None:
        log.warning(
            "extract_hidden_state_inference returned None — "
            "falling back to zero conditioning vector."
        )
        vlm_hidden = torch.zeros(CFG.vlm_hidden_dim, dtype=torch.float32)
        meta["hidden_state_fallback"] = True
    else:
        meta["hidden_state_fallback"] = False

    log.info(
        "Pipeline step 2: vlm_hidden shape=%s (%.2fs)",
        tuple(vlm_hidden.shape), meta["hidden_state_s"],
    )

    # ── Step 3: v3.0 grounded mask generation ────────────────────────────
    t3 = time.time()
    mask_np, mask_diag = bbox_to_sam2_mask(
        source_image=source_image,
        bbox_rel=bbox_rel,
        edit_description=edit_description,
        instruction=instruction,
        device=device,
    )
    meta["sam2_mask_s"]    = time.time() - t3
    meta["mask_coverage"]  = mask_diag.get("final_mask_coverage")
    meta["mask_diag"]      = mask_diag

    log.info(
        "Pipeline step 3: mask shape=%s coverage=%.3f source=%s subject=%r (%.2fs)",
        mask_np.shape, meta["mask_coverage"], mask_diag["selected_source"],
        mask_diag["subject_phrase"], meta["sam2_mask_s"],
    )

    # ── Step 4: SD inpainting (UNCHANGED) ────────────────────────────────
    t4 = time.time()
    edited_image = run_inpainting(
        source_image=source_image,
        vlm_hidden=vlm_hidden,
        instruction=edit_description if edit_description else instruction,
        mask_np=mask_np,
        device=device,
        seed=seed,
    )
    meta["inpainting_s"] = time.time() - t4

    total_s = time.time() - t0
    log.info(
        "Pipeline complete in %.2fs "
        "(vlm=%.2f hs=%.2f mask=%.2f inp=%.2f)",
        total_s,
        meta["vlm_inference_s"],
        meta["hidden_state_s"],
        meta["sam2_mask_s"],
        meta["inpainting_s"],
    )

    return {
        "edited_image": edited_image,
        "vlm_result":   vlm_result,
        "mask_np":      mask_np,
        "mask_diag":    mask_diag,
        "vlm_hidden":   vlm_hidden,
        "elapsed_s":    total_s,
        "metadata":     meta,
    }


print("run_inference_pipeline defined (v3.0).")


## §3 — Benchmark Dataset Loader

**Source:** `Benchmark.tar` is a 50.3 MB file at the root of the `sysuyy/ImgEdit` HuggingFace repo. It is **not** an HF dataset split — do not use `load_dataset(split=...)`. We download it with `huggingface_hub.hf_hub_download()` and extract it with `tarfile`.

**The repo also contains:**
- `ImgEdit_Judge/` — a fine-tuned judge model (we use Qwen2.5-VL base instead in §7)
- `all_dataset_gpt_score.json` — GPT-4V scores for the full training set (not the benchmark)
- `Parquet/`, `Singleturn/` — training data splits

**§3.1** downloads `Benchmark.tar` to Drive (skipped on rerun if already present).
**§3.2** extracts the tar and inspects the directory structure before parsing.
**§3.3** parses samples from the extracted files into a uniform `bench_samples` list.

**Re-run safety:** download and extraction are idempotent. `bench_n_samples = 20` is applied at parse time — no need to re-download to change the limit.

In [ ]:
# ── §3.1  Download Benchmark.tar from HuggingFace ──────────────────────────
#
# Why hf_hub_download and not load_dataset:
#   Benchmark.tar lives at the ROOT of the repo as a raw file, not as an HF
#   dataset split. get_dataset_split_names() and load_dataset(split=...) will
#   NOT find it.
#
# The file is ~50.3 MB, so we download it in full (no streaming needed).
# It is cached to Drive so subsequent runs skip the download entirely.

from huggingface_hub import hf_hub_download
from pathlib import Path

BENCH_TAR_DRIVE = CFG.benchmark_dir / CFG.bench_tar_filename
BENCH_TAR_DRIVE.parent.mkdir(parents=True, exist_ok=True)

if BENCH_TAR_DRIVE.exists():
    print(f'Benchmark tar already on Drive: {BENCH_TAR_DRIVE}')
    print(f'  Size: {BENCH_TAR_DRIVE.stat().st_size / 1e6:.1f} MB')
else:
    print(f'Downloading {CFG.bench_tar_filename} from {CFG.hf_dataset_id} ...')
    print('  (file is ~50 MB; should take <2 min on Colab)')
    _tmp = hf_hub_download(
        repo_id=CFG.hf_dataset_id,
        filename=CFG.bench_tar_filename,
        repo_type='dataset',
        local_dir=str(CFG.benchmark_dir),   # save directly to Drive
        local_dir_use_symlinks=False,        # write real file, not a symlink
    )
    # hf_hub_download returns the actual path; verify it's where we expect.
    _tmp = Path(_tmp)
    if _tmp != BENCH_TAR_DRIVE:
        import shutil
        shutil.move(str(_tmp), str(BENCH_TAR_DRIVE))
    print(f'Download complete. Size: {BENCH_TAR_DRIVE.stat().st_size / 1e6:.1f} MB')

assert BENCH_TAR_DRIVE.exists(), f'Expected tar at {BENCH_TAR_DRIVE}'
BENCH_TAR_PATH = BENCH_TAR_DRIVE
print(f'BENCH_TAR_PATH = {BENCH_TAR_PATH}')


In [ ]:
# ── §3.2  Extract and inspect Benchmark.tar ─────────────────────────────────
#
# We extract to CFG.benchmark_extracted_dir on Drive.
# Extraction is skipped if the dir already contains files.
#
# After extraction we do a quick structure inspection:
#   - List top-level entries
#   - Find all JSON files (likely the manifest/annotations)
#   - Count image files
# This tells us the schema BEFORE we commit to a parsing strategy.

import tarfile, json
from pathlib import Path

BENCH_EXT_DIR = CFG.benchmark_extracted_dir
BENCH_EXT_DIR.mkdir(parents=True, exist_ok=True)

# ── Extract (idempotent) ──────────────────────────────────────────────────────
existing = list(BENCH_EXT_DIR.iterdir())
if existing:
    print(f'Already extracted ({len(existing)} top-level entries). Skipping extraction.')
else:
    print(f'Extracting {BENCH_TAR_PATH.name} -> {BENCH_EXT_DIR} ...')
    with tarfile.open(BENCH_TAR_PATH, 'r:*') as tf:
        # Safety: strip any absolute paths or '..' components.
        members = []
        for m in tf.getmembers():
            if m.name.startswith('/') or '..' in m.name:
                log.warning('Skipping unsafe tar member: %s', m.name)
                continue
            members.append(m)
        tf.extractall(path=BENCH_EXT_DIR, members=members)
    print(f'Extraction complete.')

# ── Inspect structure ─────────────────────────────────────────────────────────
print(f'\nBenchmark directory structure:')
all_files = sorted(BENCH_EXT_DIR.rglob('*'))
dirs  = [f for f in all_files if f.is_dir()]
files = [f for f in all_files if f.is_file()]
jsons = [f for f in files if f.suffix.lower() == '.json']
imgs  = [f for f in files if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')]

print(f'  Directories : {len(dirs)}')
print(f'  Total files : {len(files)}')
print(f'  JSON files  : {len(jsons)}')
print(f'  Image files : {len(imgs)}')

# Top-level entries
top_level = sorted({f.relative_to(BENCH_EXT_DIR).parts[0] for f in all_files})
print(f'\nTop-level entries ({len(top_level)}):')
for entry in top_level[:30]:
    p = BENCH_EXT_DIR / entry
    desc = 'DIR' if p.is_dir() else f'{p.stat().st_size/1e3:.1f} kB'
    print(f'  {entry}  [{desc}]')

# Show JSON file paths
if jsons:
    print(f'\nJSON files found:')
    for j in jsons[:10]:
        print(f'  {j.relative_to(BENCH_EXT_DIR)}  ({j.stat().st_size/1e3:.1f} kB)')

# Preview first JSON to understand schema
if jsons:
    print(f'\nFirst JSON preview ({jsons[0].name}):')
    _sample_json = None
    file_content = ""
    try:
        with open(jsons[0], 'r', encoding='utf-8') as f:
            file_content = f.read()
            _sample_json = json.loads(file_content)
    except json.JSONDecodeError as e:
        if "Extra data" in str(e):
            print(f"  Warning: JSONDecodeError: Extra data. The file might contain multiple concatenated JSON objects or be in JSONL format.")
            # Try to load the first non-empty line as a JSON object
            first_line_parsed = False
            for line in file_content.splitlines():
                line = line.strip()
                if line:
                    try:
                        _sample_json = json.loads(line)
                        print(f"  Successfully parsed the first non-empty line as a JSON object.")
                        first_line_parsed = True
                        break
                    except json.JSONDecodeError:
                        # This line is not a complete JSON object, continue
                        pass
            if not first_line_parsed:
                print(f"  Could not parse any single line as a JSON object.")
                _sample_json = None
        else:
            print(f"  Failed to load JSON: {e}")
            _sample_json = None
    except Exception as e:
        print(f"  An unexpected error occurred while loading JSON: {e}")
        _sample_json = None


    if _sample_json is not None:
        if isinstance(_sample_json, list):
            print(f'  Type: list of {len(_sample_json)} entries')
            # Ensure the list is not empty before trying to access index 0
            if _sample_json:
                print(f'  First entry keys: {list(_sample_json[0].keys()) if isinstance(_sample_json[0], dict) else "not a dict"}')
                # Print first entry (non-image values)
                for k, v in _sample_json[0].items():
                    if not isinstance(v, (bytes, bytearray, list, dict)): # Avoid printing very long lists/dicts
                        print(f'    {k}: {str(v)[:120]}')
                    elif isinstance(v, (list, dict)):
                        print(f'    {k}: {type(v).__name__} (len={len(v)})')
            else:
                print(f'  First entry keys: empty list')
        elif isinstance(_sample_json, dict):
            print(f'  Type: dict with keys: {list(_sample_json.keys())}')
            # Print first 5 key-value pairs
            for k, v in list(_sample_json.items())[:5]:
                if not isinstance(v, (bytes, bytearray, list, dict)):
                    print(f'    {k}: {str(v)[:120]}')
                elif isinstance(v, (list, dict)):
                    print(f'    {k}: {type(v).__name__} (len={len(v)})')
        else:
            print(f'  Type: {type(_sample_json)}')
    else:
        print("  Could not parse JSON for preview.")


### §3.3 — Parse Singleturn Benchmark Samples

Parses `Benchmark/singleturn/singleturn.json` into a uniform `bench_samples` list.

**Singleturn manifest schema:**
```json
{ "<key>": { "id": "<category>/<filename>.jpg", "prompt": "...", "edit_type": "..." } }
```

- `id` is `<category>/<filename>.jpg` relative to `Benchmark/singleturn/`.
- **No ground-truth edited images** exist in this split — `has_gt=False` for all samples.
- Pixel metrics (CLIP-I, LPIPS, SSIM) are therefore skipped; LLM judge is the primary signal.
- Also loads `judge_prompt.json` which provides per-edit-type rubrics for §7.

**Edit-type filter (`CFG.bench_edit_type_filter`):** When set (default `('adjust',)`), only samples whose `edit_type` is in the tuple are retained after parsing/cache load. The cache is keyed to the filter value so changing it forces a re-parse. Set to `None` to evaluate all edit types.

**Output:** `bench_samples` — list of dicts:
```
sample_id, edit_instruction, edit_type,
source_img_path (absolute Path), target_img_path=None, has_gt=False
```

**Global side-effect:** `JUDGE_PROMPTS` dict keyed by edit_type (loaded from `judge_prompt.json`).


In [ ]:
# ── §3.3  Parse singleturn benchmark samples ────────────────────────────────
#
# Why singleturn-specific: the generic manifest parser (v1.0) attempted to
# handle unknown structures; now that we know the exact format, a dedicated
# parser is safer, clearer, and fails loudly on any mismatch.
#
# Singleturn manifest: Benchmark/singleturn/singleturn.json
#   { "<key>": { "id": "<category>/<filename>.jpg",
#                "prompt": "...",
#                "edit_type": "..." } }
#
# Judge rubrics: Benchmark/singleturn/judge_prompt.json
#   { "<edit_type>": "<rubric string>" }  — one entry per edit_type
#
# has_gt is ALWAYS False for singleturn (no GT edited images in this split).

import json
from pathlib import Path

# ── Locate the singleturn manifest ───────────────────────────────────────────
SPLIT_DIR     = CFG.benchmark_split_dir   # .../extracted/Benchmark/singleturn
MANIFEST_PATH = SPLIT_DIR / 'singleturn.json'
JUDGE_PROMPT_PATH = SPLIT_DIR / 'judge_prompt.json'
BENCH_CACHE   = CFG.benchmark_dir / f'bench_cache_{CFG.bench_split}.json'

assert CFG.bench_split == 'singleturn', (
    f"This notebook (v1.1) only supports bench_split='singleturn'. "
    f"Got '{CFG.bench_split}'. Set CFG.bench_split = 'singleturn' in §0.1."
)

if not SPLIT_DIR.exists():
    raise FileNotFoundError(
        f'Singleturn split directory not found: {SPLIT_DIR}\n'
        'Run §3.2 (extract Benchmark.tar) before this cell.'
    )
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'singleturn.json not found at {MANIFEST_PATH}\n'
        'Expected path inside extracted tar: Benchmark/singleturn/singleturn.json'
    )

# ── Load per-type judge rubrics ───────────────────────────────────────────────
# These are used in §7.2 instead of the generic prompt when
# CFG.bench_use_type_specific_judge=True.
if JUDGE_PROMPT_PATH.exists():
    with open(JUDGE_PROMPT_PATH) as _f:
        JUDGE_PROMPTS = json.load(_f)
    print(f'Loaded judge_prompt.json: {sorted(JUDGE_PROMPTS.keys())}')
else:
    JUDGE_PROMPTS = {}
    log.warning(
        'judge_prompt.json not found at %s. '
        'Type-specific judge prompts will fall back to the generic template.',
        JUDGE_PROMPT_PATH,
    )

# ── Parse the singleturn manifest ────────────────────────────────────────────
def _parse_singleturn_manifest(manifest_path: Path, split_dir: Path, n) -> list:
    """Parse bench_samples from Benchmark/singleturn/singleturn.json.

    Each entry maps a key string to:
        { "id": "<category>/<filename>.jpg",
          "prompt": "<edit instruction>",
          "edit_type": "<type>" }

    The source image path is: split_dir / entry['id']
    No ground-truth images exist; has_gt is always False.
    """
    with open(manifest_path) as f:
        raw = json.load(f)

    if not isinstance(raw, dict):
        raise ValueError(
            f'Expected singleturn.json to be a dict, got {type(raw).__name__}. '
            'Check the file is the correct singleturn manifest.'
        )

    log.info('singleturn.json has %d entries.', len(raw))

    # Validate required fields on the first entry (fast fail)
    _sample_key = next(iter(raw))
    _sample_entry = raw[_sample_key]
    for required in ('id', 'prompt', 'edit_type'):
        if required not in _sample_entry:
            raise KeyError(
                f'singleturn.json entry missing required field "{required}". '
                f'Keys found: {list(_sample_entry.keys())}'
            )

    samples = []
    parse_errors = 0

    for key, entry in raw.items():
        if n is not None and len(samples) >= n:
            break

        img_rel = entry.get('id', '')
        prompt  = entry.get('prompt', '')
        etype   = entry.get('edit_type', 'unknown')

        # Resolve source image path
        src_path = split_dir / img_rel
        if not src_path.exists():
            log.warning(
                'Key %s: source image not found: %s', key, src_path
            )
            parse_errors += 1
            continue

        if not prompt.strip():
            log.warning('Key %s: empty prompt.', key)

        # Use the manifest key as sample_id for traceability
        samples.append({
            'sample_id':        f'st_{key}',   # e.g. st_1082
            'edit_instruction': prompt,
            'edit_type':        etype,
            'source_img_path':  str(src_path),
            'target_img_path':  None,           # no GT in singleturn split
            'has_gt':           False,
        })

    if parse_errors:
        log.warning('%d entries skipped (image not found).', parse_errors)

    return samples


# ── Load from cache or parse fresh ───────────────────────────────────────────
if BENCH_CACHE.exists():
    with open(BENCH_CACHE) as f:
        cached = json.load(f)
    # Cache is valid if all source images still exist and we have enough samples.
    valid = [s for s in cached if Path(s['source_img_path']).exists()]
    need  = CFG.bench_n_samples or 0
    if len(valid) >= need:
        log.info('Cache hit (%s): %d valid samples.', BENCH_CACHE.name, len(valid))
        bench_samples = valid[:CFG.bench_n_samples] if CFG.bench_n_samples else valid
    else:
        log.info('Cache stale or insufficient (%d < %d). Re-parsing.', len(valid), need)
        bench_samples = None
else:
    bench_samples = None

if bench_samples is None:
    bench_samples = _parse_singleturn_manifest(
        MANIFEST_PATH, SPLIT_DIR, CFG.bench_n_samples
    )
    assert len(bench_samples) > 0, (
        f'No samples parsed from {MANIFEST_PATH}. '
        'Check that §3.2 extracted Benchmark.tar correctly.'
    )
    with open(BENCH_CACHE, 'w') as f:
        json.dump(bench_samples, f, indent=2)
    log.info('Cache saved: %d samples -> %s', len(bench_samples), BENCH_CACHE)

# ── Apply edit_type filter (v1.4) ────────────────────────────────────────────
#
# Why filter here (after cache load): the cache stores ALL parsed samples so
# that switching filter values does not require re-hitting the filesystem.
# We apply the filter in-memory after loading and BEFORE the coverage check.
#
# CFG.bench_edit_type_filter=None  → no filter, all types kept (v1.3 compat)
# CFG.bench_edit_type_filter=('adjust',) → only 'adjust' samples

if CFG.bench_edit_type_filter is not None:
    _filter_set = set(CFG.bench_edit_type_filter)
    _before = len(bench_samples)
    bench_samples = [s for s in bench_samples if s['edit_type'] in _filter_set]
    _after = len(bench_samples)
    log.info(
        'Edit-type filter %s applied: %d → %d samples.',
        CFG.bench_edit_type_filter, _before, _after,
    )
    if _after == 0:
        raise RuntimeError(
            f'bench_edit_type_filter={CFG.bench_edit_type_filter!r} matched 0 samples. '
            f'Types available in cache: '
            f'{sorted(set(s["edit_type"] for s in json.load(open(BENCH_CACHE))))}. '
            'Check CFG.bench_edit_type_filter or set it to None.'
        )
    print(
        f'Edit-type filter {CFG.bench_edit_type_filter}: '
        f'{_before} → {_after} samples retained.'
    )
else:
    print('No edit-type filter applied (CFG.bench_edit_type_filter=None).')

# ── Sanity-check edit_type coverage ──────────────────────────────────────────
from collections import Counter
found_types  = set(s['edit_type'] for s in bench_samples)
expected_types = set(CFG.singleturn_edit_types)
unknown_types  = found_types - expected_types
if unknown_types:
    log.warning(
        'Unexpected edit_type(s) in parsed samples: %s. '
        'Update CFG.singleturn_edit_types if these are valid new types.',
        unknown_types,
    )
missing_types = expected_types - found_types
if missing_types and (CFG.bench_n_samples is not None):
    log.info(
        'Edit types not present in this sample slice (expected with n=%d): %s',
        CFG.bench_n_samples, missing_types,
    )

# ── Final summary ─────────────────────────────────────────────────────────────
print(f'\nLoaded {len(bench_samples)} singleturn benchmark samples.')
print(f'  has_gt        : always False (no GT edited images in this split)')
print(f'  Judge rubrics : {len(JUDGE_PROMPTS)} types loaded from judge_prompt.json')
print()
print('Edit type distribution:')
for et, cnt in Counter(s['edit_type'] for s in bench_samples).most_common():
    has_rubric = '✓ rubric' if et in JUDGE_PROMPTS else '✗ no rubric'
    print(f'  {et:<16} {cnt:>4}  {has_rubric}')
if bench_samples:
    s0 = bench_samples[0]
    print()
    print(f'  sample_id         : {s0["sample_id"]}')
    print(f'  edit_instruction  : {s0["edit_instruction"][:80]}')
    print(f'  edit_type         : {s0["edit_type"]}')
    print(f'  source_img_path   : {Path(s0["source_img_path"]).name}')
    print(f'  target_img_path   : None  (no GT in singleturn split)')


## §4 — Benchmark Audit

Validates loaded samples before running expensive inference. Hard-fails on critical errors. Warns on recoverable issues.

In [ ]:
# ── §4.1  Benchmark data audit ──────────────────────────────────────────────
from collections import Counter
from PIL import Image

audit_errors   = []
audit_warnings = []
edit_type_dist = Counter()
has_gt_count   = 0

for s in bench_samples:
    sid = s['sample_id']

    src_path = Path(s['source_img_path'])
    if not src_path.exists():
        audit_errors.append(f'{sid}: source image missing at {src_path}')
        continue
    try:
        src_img = Image.open(src_path).convert('RGB')
        W, H = src_img.size
        if W < 64 or H < 64:
            audit_warnings.append(f'{sid}: very small source image {W}x{H}')
        if W > 4096 or H > 4096:
            audit_warnings.append(f'{sid}: very large source image {W}x{H} (will resize)')
    except Exception as e:
        audit_errors.append(f'{sid}: cannot open source image: {e}')
        continue

    if s['has_gt'] and s['target_img_path']:
        tgt_path = Path(s['target_img_path'])
        if not tgt_path.exists():
            audit_warnings.append(f'{sid}: has_gt=True but file missing; marking False')
            s['has_gt'] = False
        else:
            try:
                tgt_img = Image.open(tgt_path).convert('RGB')
                tW, tH = tgt_img.size
                if (tW, tH) != (W, H):
                    audit_warnings.append(
                        f'{sid}: source {W}x{H} != target {tW}x{tH} '
                        '(will resize target for pixel metrics)'
                    )
                has_gt_count += 1
            except Exception as e:
                audit_warnings.append(f'{sid}: bad target image: {e}; marking has_gt=False')
                s['has_gt'] = False

    if not s['edit_instruction'].strip():
        audit_warnings.append(f'{sid}: empty edit_instruction')

    edit_type_dist[s['edit_type']] += 1

id_counts = Counter(s['sample_id'] for s in bench_samples)
dups = [sid for sid, cnt in id_counts.items() if cnt > 1]
if dups:
    audit_errors.append(f'Duplicate sample_ids: {dups}')

gt_frac = has_gt_count / max(len(bench_samples), 1)
if gt_frac < 0.5:
    audit_warnings.append(
        f'Only {has_gt_count}/{len(bench_samples)} samples have GT ({gt_frac*100:.1f}%). '
        'LPIPS/SSIM/CLIP-I will be skipped for samples without GT.'
    )

print(f'Audit complete: {len(bench_samples)} samples.')
print(f'  Errors   : {len(audit_errors)}')
print(f'  Warnings : {len(audit_warnings)}')
print(f'  Has GT   : {has_gt_count}/{len(bench_samples)} ({gt_frac*100:.1f}%)')
print()
print('Edit type distribution:')
for et, cnt in edit_type_dist.most_common():
    print(f'  {et:<18} {cnt:>4}')
if audit_warnings:
    print('\nWarnings (first 20):')
    for w in audit_warnings[:20]:
        print(f'  WARN: {w}')
if audit_errors:
    print('\nErrors:')
    for e in audit_errors[:20]:
        print(f'  ERROR: {e}')
    raise RuntimeError(
        f'Audit failed with {len(audit_errors)} error(s). Fix before running §5.'
    )
print('\nAudit passed. Safe to proceed to §5.')


## §5 — Run Benchmark Inference

Runs `run_inference_pipeline()` (loaded in §2) on each benchmark sample.

**Resume safety:** completed samples are skipped. Results written after each sample.

In [ ]:
# ── §5.1  Pre-flight: verify Phase 3 inference pipeline is loaded ─────────────
# Variable names must match what each loader cell actually assigns:
#   sam2_predictor  ← §21.2   clip          ← §23.1   sd_tokenizer  ← §23.1
#   noise_scheduler ← §23.1   vlm_adapter   ← §23.2
_required = [
    'vlm_model', 'vlm_processor',
    'grounding_model', 'grounding_processor',
    'sam2_predictor',
    'vae', 'unet', 'clip', 'sd_tokenizer', 'noise_scheduler',
    'vlm_adapter',
    'run_inference_pipeline',
]
_missing = [g for g in _required if g not in globals()]
if _missing:
    raise RuntimeError(
        f'Missing globals: {_missing}.\n'
        'Run §2 cells (Phase 3 model loading) before this cell.'
    )
print('All model globals present. Ready for benchmark inference.')


In [ ]:
# ── §5.2  Benchmark inference loop ──────────────────────────────────────────
#
# Per-sample output:
#   <sid>_edited.jpg   model output
#   <sid>_source.jpg   copy of source for convenience
#   <sid>_mask.png     localisation mask
#
# Per-sample record (appended to bench_inference_results.json):
#   sample_id, edit_instruction, edit_type, has_gt, target_img_path
#   edited_img_path, source_img_path, mask_img_path
#   status: 'ok' | 'error'
#   vlm_result, mask_coverage, elapsed_s, timing, grounding diagnostics

import json, time, traceback
from pathlib import Path
from PIL import Image

out_dir = CFG.outputs_bench
out_dir.mkdir(parents=True, exist_ok=True)

if CFG.bench_results_path.exists():
    with open(CFG.bench_results_path) as f:
        bench_inference_results = json.load(f)
    done_ids = {r['sample_id'] for r in bench_inference_results if r.get('status') == 'ok'}
    print(f'Resuming: {len(done_ids)} samples already completed.')
else:
    bench_inference_results = []
    done_ids = set()

print(f'Samples to run: {len(bench_samples)}')
print(f'Output dir    : {out_dir}')

for idx, sample in enumerate(bench_samples):
    sid = sample['sample_id']

    if sid in done_ids:
        print(f'[{idx+1}/{len(bench_samples)}] SKIP {sid}')
        continue

    src_path = Path(sample['source_img_path'])
    if not src_path.exists():
        bench_inference_results.append({
            'sample_id': sid, 'status': 'error',
            'error_msg': f'source image missing: {src_path}',
        })
        continue

    print(f'[{idx+1}/{len(bench_samples)}] {sid}: {sample["edit_instruction"][:60]} ...')

    try:
        src_img = Image.open(src_path).convert('RGB')

        result = run_inference_pipeline(
            source_image=src_img,
            instruction=sample['edit_instruction'],
            seed=42,
        )

        edited_path = out_dir / f'{sid}_edited.jpg'
        mask_path   = out_dir / f'{sid}_mask.png'
        src_out     = out_dir / f'{sid}_source.jpg'

        result['edited_image'].save(edited_path, quality=95)
        src_img.save(src_out, quality=95)
        Image.fromarray(result['mask_np']).save(mask_path)

        vlm_r = result['vlm_result']
        diag  = result.get('mask_diag', {})

        record = {
            'sample_id':        sid,
            'edit_instruction': sample['edit_instruction'],
            'edit_type':        sample['edit_type'],
            'has_gt':           sample['has_gt'],
            'target_img_path':  sample['target_img_path'],
            'edited_img_path':  str(edited_path),
            'source_img_path':  str(src_out),
            'mask_img_path':    str(mask_path),
            'status':           'ok',
            'vlm_result': {
                'edit_type':        vlm_r.get('edit_type'),
                'bbox':             vlm_r.get('bbox'),
                'edit_description': vlm_r.get('edit_description', ''),
                'used_fallback':    vlm_r.get('used_fallback', False),
            },
            'mask_coverage':        result['metadata'].get('mask_coverage'),
            'elapsed_s':            result['elapsed_s'],
            'timing':               {
                k: v for k, v in result['metadata'].items()
                if isinstance(v, (int, float))
            },
            'selected_source':      diag.get('selected_source'),
            'subject_phrase':       diag.get('subject_phrase'),
            'grounding_n_boxes':    diag.get('grounding_n_boxes'),
            'grounding_top_score':  diag.get('grounding_top_score'),
        }

    except Exception:
        record = {
            'sample_id': sid,
            'edit_type': sample.get('edit_type', 'unknown'),
            'has_gt':    sample.get('has_gt', False),
            'status':    'error',
            'error_msg': traceback.format_exc()[-800:],
        }
        log.error('Sample %s failed.', sid)

    bench_inference_results.append(record)
    done_ids.add(sid)

    # Write after every sample for restart safety
    with open(CFG.bench_results_path, 'w') as f:
        json.dump(bench_inference_results, f, indent=2)

ok_n  = sum(1 for r in bench_inference_results if r.get('status') == 'ok')
err_n = sum(1 for r in bench_inference_results if r.get('status') == 'error')
print(f'\nInference complete.  OK={ok_n}  Errors={err_n}')
print(f'Results: {CFG.bench_results_path}')


## §6 — Automated Image Metrics

Three reference-based metrics computed against ground-truth (skipped when `has_gt=False`):

| Metric | What it measures | Better |
|--------|-----------------|--------|
| **CLIP-I** | Semantic similarity via CLIP embeddings | Higher |
| **LPIPS** | Perceptual distance (AlexNet features) | Lower |
| **SSIM** | Structural similarity | Higher |

All images resized to 512×512 before metric computation for comparability.

In [ ]:
# ── §6.1  Load CLIP for CLIP-I metric ───────────────────────────────────────
import torch
from transformers import CLIPProcessor, CLIPModel

print(f'Loading CLIP: {CFG.clip_judge_model_id} ...')
clip_model = CLIPModel.from_pretrained(
    CFG.clip_judge_model_id,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CFG.clip_judge_model_id)
print(f'CLIP loaded. Projection dim: {clip_model.config.projection_dim}')


@torch.no_grad()
def clip_image_similarity(img_a: Image.Image, img_b: Image.Image) -> float:
    """Cosine similarity between CLIP image embeddings. Range [-1,1], higher=better."""
    inputs = clip_processor(images=[img_a, img_b], return_tensors='pt').to(DEVICE)
    feats = clip_model.get_image_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return float((feats[0] * feats[1]).sum().item())


# Smoke test: self-similarity must be ~1.0
_d = Image.new('RGB', (64, 64), (100, 100, 100))
_s = clip_image_similarity(_d, _d)
assert abs(_s - 1.0) < 0.01, f'CLIP-I self-similarity = {_s}, expected ~1.0'
print(f'CLIP-I smoke test passed (self-sim = {_s:.4f})')


In [ ]:
# ── §6.2  LPIPS ──────────────────────────────────────────────────────────────
import lpips, torch, numpy as np

lpips_fn = lpips.LPIPS(net='alex').to(DEVICE).eval()
print('LPIPS (AlexNet) loaded.')


def _to_lpips(img: Image.Image, size: int = 512) -> torch.Tensor:
    """PIL -> LPIPS input tensor. LPIPS expects float32 in [-1,1], shape (1,3,H,W)."""
    arr = np.array(img.resize((size, size), Image.LANCZOS).convert('RGB'), dtype=np.float32)
    arr = arr / 127.5 - 1.0
    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(DEVICE)


@torch.no_grad()
def compute_lpips(edited: Image.Image, target: Image.Image, size: int = 512) -> float:
    """LPIPS perceptual distance. Lower = more perceptually similar to GT."""
    t_ed = _to_lpips(edited, size)
    t_gt = _to_lpips(target, size)
    assert t_ed.shape == t_gt.shape, f'Shape mismatch: {t_ed.shape} vs {t_gt.shape}'
    return float(lpips_fn(t_ed, t_gt).item())


_d = Image.new('RGB', (64, 64), (100, 100, 100))
assert compute_lpips(_d, _d) < 0.05, 'Self-LPIPS should be ~0'
print('LPIPS smoke test passed.')


In [ ]:
# ── §6.3  SSIM ───────────────────────────────────────────────────────────────
import numpy as np
from skimage.metrics import structural_similarity as skimage_ssim


def compute_ssim(edited: Image.Image, target: Image.Image, size: int = 512) -> float:
    """SSIM in [0,1]. Higher = better structural similarity to GT."""
    a = np.array(edited.resize((size, size), Image.LANCZOS).convert('RGB'))
    b = np.array(target.resize((size, size), Image.LANCZOS).convert('RGB'))
    assert a.shape == b.shape, f'SSIM shape mismatch: {a.shape} vs {b.shape}'
    return float(skimage_ssim(a, b, channel_axis=2, data_range=255))


_d = Image.new('RGB', (64, 64), (100, 100, 100))
assert abs(compute_ssim(_d, _d) - 1.0) < 1e-4, 'Self-SSIM should be 1.0'
print('SSIM smoke test passed.')


In [ ]:
# ── §6.4  Run automated metrics ──────────────────────────────────────────────
import json, time
from pathlib import Path

with open(CFG.bench_results_path) as f:
    bench_inference_results = json.load(f)

if CFG.bench_scores_path.exists():
    with open(CFG.bench_scores_path) as f:
        auto_scores = json.load(f)
    scored_ids = {s['sample_id'] for s in auto_scores}
    print(f'Resuming: {len(scored_ids)} already scored.')
else:
    auto_scores = []
    scored_ids  = set()

for r in bench_inference_results:
    sid = r['sample_id']
    if sid in scored_ids:
        continue

    rec = {
        'sample_id': sid,
        'edit_type': r.get('edit_type', 'unknown'),
        'has_gt':    r.get('has_gt', False),
        'status':    r.get('status'),
        'clip_i': None, 'lpips': None, 'ssim': None,
        'metrics_error': None,
    }

    if r.get('status') != 'ok':
        auto_scores.append(rec)
        scored_ids.add(sid)
        continue

    edited_path = Path(r['edited_img_path'])
    target_path = Path(r['target_img_path']) if r.get('target_img_path') else None

    if not edited_path.exists():
        rec['metrics_error'] = f'edited image missing: {edited_path}'
        auto_scores.append(rec)
        scored_ids.add(sid)
        continue

    try:
        edited_img = Image.open(edited_path).convert('RGB')

        if r.get('has_gt') and target_path and target_path.exists():
            tgt = Image.open(target_path).convert('RGB')
            e512 = edited_img.resize((512, 512), Image.LANCZOS)
            t512 = tgt.resize((512, 512), Image.LANCZOS)
            rec['clip_i'] = clip_image_similarity(e512, t512)
            rec['lpips']  = compute_lpips(edited_img, tgt)
            rec['ssim']   = compute_ssim(edited_img, tgt)
            print(
                f'{sid}: CLIP-I={rec["clip_i"]:.3f}  '
                f'LPIPS={rec["lpips"]:.3f}  '
                f'SSIM={rec["ssim"]:.3f}'
            )
        else:
            print(f'{sid}: no GT — pixel metrics skipped.')
    except Exception as e:
        rec['metrics_error'] = str(e)
        log.error('Metrics failed for %s: %s', sid, e)

    auto_scores.append(rec)
    scored_ids.add(sid)

    with open(CFG.bench_scores_path, 'w') as f:
        json.dump(auto_scores, f, indent=2)

ok_sc = [s for s in auto_scores if s['clip_i'] is not None]
print(f'\nAuto-metrics done. Scored with GT: {len(ok_sc)}/{len(auto_scores)}')
if ok_sc:
    print(f'  Mean CLIP-I : {sum(s["clip_i"] for s in ok_sc)/len(ok_sc):.4f}')
    print(f'  Mean LPIPS  : {sum(s["lpips"]  for s in ok_sc)/len(ok_sc):.4f}')
    print(f'  Mean SSIM   : {sum(s["ssim"]   for s in ok_sc)/len(ok_sc):.4f}')


## §7 — LLM Judge Scoring

Uses **Qwen2.5-VL** as a vision-language judge to score each edited image on four criteria:

| Criterion | Description | Range |
|-----------|-------------|-------|
| **edit_correctness** | Was the requested edit applied correctly? | 1–5 |
| **edit_quality** | Is the edited region visually realistic and artifact-free? | 1–5 |
| **localization** | Is the edit applied to the right area? | 1–5 |
| **preservation** | Is the unedited region unchanged? | 1–5 |

**Model choice:** `judge_use_lora=False` (default) uses the base Qwen2.5-VL — less biased since the fine-tuned model was trained on the same distribution it is judging.

The judge is called twice per sample when `has_gt=True`: once without GT (blind evaluation) and once with GT shown as reference. Set `JUDGE_USE_GT=False` to skip the second call.

In [ ]:
# ── §7.1  Load judge model ───────────────────────────────────────────────────
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading judge model: {CFG.vlm_model_id} (use_lora={CFG.judge_use_lora}) ...')
judge_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG.vlm_model_id,
    quantization_config=_bnb,
    device_map='auto',
    torch_dtype=torch.float16,
)

if CFG.judge_use_lora:
    if not CFG.ckpt_phase1.exists():
        raise RuntimeError(
            f'Phase 1 LoRA checkpoint not found at {CFG.ckpt_phase1}. '
            'Set CFG.judge_use_lora = False to use the base model.'
        )
    print(f'  Attaching Phase 1 LoRA from: {CFG.ckpt_phase1}')
    judge_model = PeftModel.from_pretrained(judge_model, CFG.ckpt_phase1)
    judge_model = judge_model.merge_and_unload()

judge_model.eval()

# Reuse vlm_processor if Phase 3 models are loaded; otherwise load fresh.
if 'vlm_processor' in globals():
    judge_processor = vlm_processor
    print('  Reusing vlm_processor from Phase 3.')
else:
    judge_processor = AutoProcessor.from_pretrained(
        CFG.vlm_model_id,
        min_pixels=256*28*28,
        max_pixels=CFG.phase1_max_img_pixels,
    )

print('Judge model ready.')


In [ ]:
# ── §7.2  Judge prompt template and output parser ────────────────────────────
#
# v1.1: When CFG.bench_use_type_specific_judge=True (default), we use the
# edit-type-specific rubric from JUDGE_PROMPTS (loaded from judge_prompt.json
# in §3.3) as the user-facing prompt.  This matches the rubric design of the
# ImgEdit benchmark authors rather than our generic four-criterion template.
#
# The four generic criteria (edit_correctness, edit_quality, localization,
# preservation) are STILL extracted via regex/JSON parsing from the model
# output — they map onto the rubric's three named dimensions (which vary by
# type) by requesting a fixed JSON block at the end of the type-specific
# prompt.  This keeps scoring comparable across types.
#
# IMPORTANT: this judge template is intentionally SEPARATE from the Phase 1
# training template. Never mix them.

import json, re
from typing import Optional, Dict

# ── Generic fallback (used when no type-specific rubric exists) ───────────────
JUDGE_SYSTEM_PROMPT_GENERIC = (
    'You are an expert image editing evaluator. '
    'You will be shown a source image, an edited image, and the edit instruction. '
    'Score the edited image on four criteria. '
    'Think step by step, then output your scores as a JSON block.'
)

JUDGE_CRITERIA_DESC_GENERIC = (
    '1. edit_correctness (1-5): Was the requested edit correctly applied? '
    '5=perfect, 1=not applied.\n'
    '2. edit_quality (1-5): Is the edited region visually realistic? '
    '5=photorealistic, 1=severe artifacts.\n'
    '3. localization (1-5): Is the edit confined to the right region? '
    '5=perfectly localised, 1=bleeds everywhere.\n'
    '4. preservation (1-5): Is the unedited region unchanged? '
    '5=perfectly preserved, 1=major unwanted changes.'
)

# Required output block appended to every judge call.
# The scores map our four criteria onto whatever the type-specific rubric calls them.
JUDGE_OUTPUT_FORMAT = (
    '\nRegardless of the rubric above, output exactly this JSON block at the end '
    '(nothing after it):\n'
    '```json\n'
    '{"edit_correctness": <1-5>, "edit_quality": <1-5>, '
    '"localization": <1-5>, "preservation": <1-5>, '
    '"reasoning": "<brief reasoning>"}\n'
    '```\n'
    'Scores must be integers 1–5.'
)

JUDGE_CRITERIA = ['edit_correctness', 'edit_quality', 'localization', 'preservation']


def _get_type_rubric(edit_type: str) -> str:
    """Return the type-specific rubric string, or empty string if not found."""
    if not CFG.bench_use_type_specific_judge:
        return ''
    return JUDGE_PROMPTS.get(edit_type, '')


def build_judge_messages(
    source_img, edited_img, instruction: str, edit_type: str = 'unknown',
    target_img=None,
):
    """Build Qwen2.5-VL chat messages for the singleturn judge call.

    Always shows: source + edited image + instruction.
    Optionally shows ground-truth for reference (unused in singleturn since has_gt=False).

    v1.1: Uses the edit-type-specific rubric from judge_prompt.json when available.
    The generic four-criterion rubric is used as a fallback.
    The fixed JSON output block is appended regardless of rubric choice so that
    parse_judge_output() works uniformly.
    """
    content = []
    content.append({'type': 'text', 'text': 'SOURCE IMAGE:'})
    content.append({'type': 'image', 'image': source_img})
    content.append({'type': 'text', 'text': 'EDITED IMAGE (to evaluate):'})
    content.append({'type': 'image', 'image': edited_img})
    if target_img is not None:
        content.append({'type': 'text', 'text': 'GROUND TRUTH (reference only):'})
        content.append({'type': 'image', 'image': target_img})

    # ── Choose rubric ──────────────────────────────────────────────────────
    type_rubric = _get_type_rubric(edit_type)
    if type_rubric:
        # Type-specific rubric from judge_prompt.json.
        # It already describes the three scoring dimensions; we just append
        # the JSON output block to standardise extraction.
        user_text = (
            f'Edit instruction: "{instruction}"\n\n'
            f'{type_rubric.strip()}\n'
            f'{JUDGE_OUTPUT_FORMAT}'
        )
        system_prompt = (
            'You are an expert image editing evaluator. '
            'Follow the rubric below exactly. '
            'Think step by step, then output the JSON block as specified.'
        )
    else:
        # Generic fallback (edit_type not in judge_prompt.json or feature disabled).
        user_text = (
            f'Edit instruction: "{instruction}"\n\n'
            f'Criteria:\n{JUDGE_CRITERIA_DESC_GENERIC}\n\n'
            f'{JUDGE_OUTPUT_FORMAT}'
        )
        system_prompt = JUDGE_SYSTEM_PROMPT_GENERIC

    content.append({'type': 'text', 'text': user_text})
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': content},
    ]


def parse_judge_output(text: str) -> dict:
    """Extract per-criterion scores from judge generation output.

    Tries JSON block first; falls back to per-criterion regex.
    Returns None for any criterion that cannot be parsed.
    Why fail-safe: the judge model may occasionally output malformed JSON;
    we must not crash the evaluation loop.
    """
    scores = {c: None for c in JUDGE_CRITERIA}
    scores.update({'reasoning': None, 'raw_output': text})

    # JSON block extraction
    m = re.search(r'```json\s*(\{.*?\})\s*```', text, re.DOTALL)
    if m:
        try:
            parsed = json.loads(m.group(1))
            for c in JUDGE_CRITERIA:
                v = parsed.get(c)
                if v is not None:
                    v = max(CFG.judge_score_min,
                            min(CFG.judge_score_max, int(round(float(v)))))
                    scores[c] = v
            scores['reasoning'] = parsed.get('reasoning', '')
            return scores
        except (json.JSONDecodeError, ValueError):
            pass  # Fall through to regex

    # Regex fallback: scan for "criterion": <digit>
    for c in JUDGE_CRITERIA:
        m2 = re.search(rf'{c}["\'\s:]+([1-5])', text, re.IGNORECASE)
        if m2:
            scores[c] = max(CFG.judge_score_min,
                            min(CFG.judge_score_max, int(m2.group(1))))
    return scores


# ── Validation: check type-rubric coverage ───────────────────────────────────
if CFG.bench_use_type_specific_judge and bench_samples:
    types_in_samples = set(s['edit_type'] for s in bench_samples)
    types_with_rubric = set(JUDGE_PROMPTS.keys())
    missing_rubric = types_in_samples - types_with_rubric
    if missing_rubric:
        log.warning(
            'No rubric for edit type(s): %s. Generic template will be used.', missing_rubric
        )
    else:
        print(f'All {len(types_in_samples)} edit type(s) have rubrics. ✓')

print('Judge template and parser defined.')
print(f'  Criteria        : {JUDGE_CRITERIA}')
print(f'  Score range     : [{CFG.judge_score_min}, {CFG.judge_score_max}]')
print(f'  Type-specific   : {CFG.bench_use_type_specific_judge}')
print(f'  Rubric types    : {sorted(JUDGE_PROMPTS.keys())}')


In [ ]:
# ── §7.3  Judge smoke test ───────────────────────────────────────────────────
import torch
from qwen_vl_utils import process_vision_info

print('Running judge smoke test ...')

# Use first OK inference result if available; else synthetic images.
_tr = next(
    (r for r in bench_inference_results
     if r.get('status') == 'ok' and Path(r['edited_img_path']).exists()),
    None
)
if _tr:
    _src = Image.open(_tr['source_img_path']).convert('RGB')
    _ed  = Image.open(_tr['edited_img_path']).convert('RGB')
    _tgt = (
        Image.open(_tr['target_img_path']).convert('RGB')
        if _tr.get('has_gt') and _tr.get('target_img_path')
           and Path(_tr['target_img_path']).exists()
        else None
    )
    _ins = _tr['edit_instruction']
else:
    print('  No OK results yet; using synthetic images.')
    _src = Image.new('RGB', (256, 256), (100, 150, 200))
    _ed  = Image.new('RGB', (256, 256), (120, 160, 210))
    _tgt = None
    _ins = 'Make the sky brighter'

msgs = build_judge_messages(_src, _ed, _ins, _tgt)
_tp  = judge_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
_ii, _vi = process_vision_info(msgs)
_inp = judge_processor(
    text=[_tp], images=_ii, videos=_vi,
    return_tensors='pt', padding=True,
).to(DEVICE)

with torch.no_grad():
    _gen = judge_model.generate(
        **_inp,
        max_new_tokens=CFG.judge_max_new_tokens,
        do_sample=False,
        temperature=None, top_p=None,
    )

_txt = judge_processor.batch_decode(
    _gen[:, _inp['input_ids'].shape[1]:],
    skip_special_tokens=True,
)[0]

print('Judge output (first 500 chars):')
print(_txt[:500])
print()

_sc = parse_judge_output(_txt)
print('Parsed scores:')
for k in JUDGE_CRITERIA + ['reasoning']:
    print(f'  {k}: {_sc[k]}')

_n_parsed = sum(1 for c in JUDGE_CRITERIA if _sc[c] is not None)
if _n_parsed == 0:
    print('WARNING: 0/4 criteria parsed. Check judge prompt.')
else:
    print(f'Smoke test passed: {_n_parsed}/4 criteria parsed.')


In [ ]:
# ── §7.4  Run LLM judge on all benchmark samples ────────────────────────────
#
# v1.1 change: edit_type is now passed to build_judge_messages() so the
# type-specific rubric from judge_prompt.json is used.
#
# Note: has_gt=False for all singleturn samples, so the GT-aware branch
# (JUDGE_USE_GT=True) will never be entered. It is kept for forward
# compatibility if this notebook is reused with a split that has GT.

import json, time, traceback
from pathlib import Path
from PIL import Image
from qwen_vl_utils import process_vision_info

# Set False to skip the GT-aware second call (halves judge runtime).
# For singleturn this has no effect (has_gt=False for all samples).
JUDGE_USE_GT = True

# Build lookup from sample_id -> score record
score_lookup  = {s['sample_id']: s for s in auto_scores}
judge_done_ids = {s['sample_id'] for s in auto_scores if s.get('judge_done')}
print(f'Judge: {len(judge_done_ids)} already scored.')


@torch.no_grad()
def _run_judge_call(src_img, ed_img, instr: str, edit_type: str = 'unknown',
                    tgt_img=None):
    """Single judge call. Returns parsed scores dict."""
    msgs  = build_judge_messages(
        src_img, ed_img, instr, edit_type=edit_type, target_img=tgt_img
    )
    tp    = judge_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ii, vi = process_vision_info(msgs)
    inp = judge_processor(
        text=[tp], images=ii, videos=vi,
        return_tensors='pt', padding=True,
    ).to(DEVICE)
    gen = judge_model.generate(
        **inp, max_new_tokens=CFG.judge_max_new_tokens,
        do_sample=False, temperature=None, top_p=None,
    )
    txt = judge_processor.batch_decode(
        gen[:, inp['input_ids'].shape[1]:], skip_special_tokens=True
    )[0]
    return parse_judge_output(txt)


for r in bench_inference_results:
    sid = r['sample_id']
    if sid in judge_done_ids:
        continue

    # Ensure score record exists (for samples that had no auto-metric record)
    if sid not in score_lookup:
        new_rec = {
            'sample_id': sid,
            'edit_type': r.get('edit_type', 'unknown'),
            'has_gt':    r.get('has_gt', False),
            'status':    r.get('status'),
            'clip_i': None, 'lpips': None, 'ssim': None,
        }
        auto_scores.append(new_rec)
        score_lookup[sid] = new_rec

    rec = score_lookup[sid]

    if r.get('status') != 'ok':
        rec['judge_done'] = True
        judge_done_ids.add(sid)
        continue

    try:
        src_img   = Image.open(r['source_img_path']).convert('RGB')
        ed_img    = Image.open(r['edited_img_path']).convert('RGB')
        instr     = r['edit_instruction']
        edit_type = r.get('edit_type', 'unknown')

        t0 = time.time()

        # Blind evaluation (no GT).
        # In singleturn all samples have has_gt=False so this is the only call.
        j_nog = _run_judge_call(src_img, ed_img, instr, edit_type=edit_type)
        rec['judge_no_gt'] = {
            c: j_nog.get(c) for c in JUDGE_CRITERIA + ['reasoning']
        }
        rec['judge_rubric_type'] = edit_type  # record which rubric was used

        # GT-aware evaluation (skipped for singleturn; kept for forward compat)
        if JUDGE_USE_GT and r.get('has_gt') and r.get('target_img_path'):
            tp = Path(r['target_img_path'])
            if tp.exists():
                tgt_img = Image.open(tp).convert('RGB')
                j_gt = _run_judge_call(
                    src_img, ed_img, instr, edit_type=edit_type, tgt_img=tgt_img
                )
                rec['judge_with_gt'] = {
                    c: j_gt.get(c) for c in JUDGE_CRITERIA + ['reasoning']
                }

        rec['judge_done'] = True
        judge_done_ids.add(sid)

        jn = rec.get('judge_no_gt', {})
        print(
            f'{sid}  EC={jn.get("edit_correctness")} '
            f'EQ={jn.get("edit_quality")} '
            f'L={jn.get("localization")} '
            f'P={jn.get("preservation")}  '
            f'({time.time()-t0:.1f}s)'
        )

    except Exception:
        rec['judge_error'] = traceback.format_exc()[-500:]
        rec['judge_done']  = True
        judge_done_ids.add(sid)
        log.error('Judge failed for %s.', sid)

    with open(CFG.bench_scores_path, 'w') as f:
        json.dump(auto_scores, f, indent=2)

print(f'\nJudge scoring complete. Results: {CFG.bench_scores_path}')


## §8 — Results Aggregation & Report

Produces a structured report from `bench_scores.json` for the active edit-type filter (`adjust` by default in v1.4):

- Overall averages for all 4 judge criteria (edit_correctness, edit_quality, localization, preservation).
- Per-edit-type breakdown across whichever edit types are present in the filtered `bench_scores.json` (only `adjust` by default in v1.4).
- (Full 9-type breakdown — action, add, adjust, background, compose, extract, remove, replace, style.
- Pixel metrics (CLIP-I, LPIPS, SSIM) are included in the schema for completeness but will be `N/A` for all singleturn samples (no GT exists in this split).

Results are also written to `bench_report.json` for downstream comparison.


In [ ]:
# ── §8.1  Aggregate and print benchmark report ───────────────────────────────
import json
from pathlib import Path
from collections import defaultdict

with open(CFG.bench_scores_path) as f:
    final_scores = json.load(f)

ok_sc  = [s for s in final_scores if s.get('status') == 'ok']
err_sc = [s for s in final_scores if s.get('status') == 'error']


def _mean(vals):
    vals = [v for v in vals if v is not None]
    return sum(vals) / len(vals) if vals else None


def _fmt(v, fmt='.3f'):
    return f'{v:{fmt}}' if v is not None else '  N/A '


def _judge_means(recs, mode='no_gt'):
    key = f'judge_{mode}'
    buckets = {c: [] for c in JUDGE_CRITERIA}
    for s in recs:
        j = s.get(key) or {}
        for c in JUDGE_CRITERIA:
            if j.get(c) is not None:
                buckets[c].append(j[c])
    return {c: _mean(v) for c, v in buckets.items()}


print('=' * 72)
_filter_tag = (
    '/'.join(sorted(CFG.bench_edit_type_filter))
    if CFG.bench_edit_type_filter else 'all'
)
print(f'BENCHMARK REPORT — Phase 4 v1.4 (singleturn split | filter={_filter_tag})')
print('=' * 72)
gt_n = sum(1 for s in ok_sc if s.get('has_gt'))
print(f'Total        : {len(final_scores)}')
print(f'OK           : {len(ok_sc)}')
print(f'Errors       : {len(err_sc)}')
print(f'With GT      : {gt_n}  (always 0 for singleturn split)')

print()
print('── Pixel metrics (GT required — N/A for singleturn) ─────────────────')
print(f'  CLIP-I (higher better): {_fmt(_mean([s.get("clip_i") for s in ok_sc]))}')
print(f'  LPIPS  (lower  better): {_fmt(_mean([s.get("lpips")  for s in ok_sc]))}')
print(f'  SSIM   (higher better): {_fmt(_mean([s.get("ssim")   for s in ok_sc]))}')

print()
print('── LLM judge (blind, no GT) — primary metric for singleturn ─────────')
j_nog = _judge_means(ok_sc, 'no_gt')
for c in JUDGE_CRITERIA:
    print(f'  {c:<22}: {_fmt(j_nog[c])}')

# GT-aware section (will be empty for singleturn but kept for compat)
if any(s.get('judge_with_gt') for s in ok_sc):
    print()
    print('── LLM judge (GT-aware) ──────────────────────────────────────────────')
    j_gt = _judge_means(ok_sc, 'with_gt')
    for c in JUDGE_CRITERIA:
        print(f'  {c:<22}: {_fmt(j_gt[c])}')

# Per-edit-type breakdown
by_type = defaultdict(list)
for s in ok_sc:
    by_type[s.get('edit_type', 'unknown')].append(s)

if by_type:
    print()
    print('── Per-edit-type breakdown (singleturn) ──────────────────────────────')
    hdr = (
        f'{"Edit type":<16} {"N":>4}  '
        f'{"EC":>4} {"EQ":>4} {"Loc":>4} {"Pre":>4}'
    )
    print(hdr)
    print('-' * len(hdr))
    for et in sorted(by_type):
        recs = by_type[et]
        j    = _judge_means(recs, 'no_gt')
        print(
            f'{et:<16} {len(recs):>4}  '
            f'{_fmt(j["edit_correctness"], ".2f"):>4} '
            f'{_fmt(j["edit_quality"],     ".2f"):>4} '
            f'{_fmt(j["localization"],     ".2f"):>4} '
            f'{_fmt(j["preservation"],     ".2f"):>4}'
        )

# Save report
report = {
    'version':     '1.4',
    'bench_split': CFG.bench_split,
    'n_total': len(final_scores), 'n_ok': len(ok_sc), 'n_error': len(err_sc),
    'n_with_gt': gt_n,
    'overall': {
        'clip_i': _mean([s.get('clip_i') for s in ok_sc]),
        'lpips':  _mean([s.get('lpips')  for s in ok_sc]),
        'ssim':   _mean([s.get('ssim')   for s in ok_sc]),
        'judge_no_gt': j_nog,
    },
    'by_edit_type': {
        et: {
            'n':       len(recs),
            'clip_i':  _mean([s.get('clip_i') for s in recs]),
            'lpips':   _mean([s.get('lpips')  for s in recs]),
            'ssim':    _mean([s.get('ssim')   for s in recs]),
            'judge':   _judge_means(recs, 'no_gt'),
        }
        for et, recs in by_type.items()
    },
}
report_path = CFG.outputs_bench / 'bench_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'\nReport saved: {report_path}')


## §9 — Verification

Sanity-checks score distributions before trusting results:
- All metric values must be in valid ranges.
- Judge parse-failure rate must be below 30%.
- Warns if all scores are suspiciously perfect (prompt-injection check).

This is the final gate. If it passes, the benchmark run is complete.

In [ ]:
# ── §9.1  Verification: sanity-check score distributions ──────────────────
#
# v1.1 note: CLIP-I/LPIPS/SSIM will all be None for singleturn (no GT).
# The pixel-metric range checks are skipped when values are None, so
# the verification cell works correctly without modification.
# The primary check is judge parse-failure rate and score range.
──
import json
from pathlib import Path

with open(CFG.bench_scores_path) as f:
    v_sc = json.load(f)

v_ok = [s for s in v_sc if s.get('status') == 'ok']

verify_errors   = []
verify_warnings = []

# Range checks
for s in v_ok:
    sid = s['sample_id']
    ci = s.get('clip_i')
    lp = s.get('lpips')
    ss = s.get('ssim')
    if ci is not None and not (-1.0 <= ci <= 1.0):
        verify_errors.append(f'{sid}: CLIP-I={ci} out of [-1,1]')
    if lp is not None and lp < 0:
        verify_errors.append(f'{sid}: LPIPS={lp} negative')
    if ss is not None and not (0.0 <= ss <= 1.0):
        verify_errors.append(f'{sid}: SSIM={ss} out of [0,1]')
    for mode in ('judge_no_gt', 'judge_with_gt'):
        j = s.get(mode) or {}
        for c in JUDGE_CRITERIA:
            v = j.get(c)
            if v is not None and not (CFG.judge_score_min <= v <= CFG.judge_score_max):
                verify_errors.append(
                    f'{sid}/{mode}/{c}={v} out of [{CFG.judge_score_min},{CFG.judge_score_max}]'
                )

# Parse-failure rate
judge_ok = [s for s in v_ok if s.get('judge_done') and not s.get('judge_error')]
fully_parsed = [
    s for s in judge_ok
    if s.get('judge_no_gt') and
    all(s['judge_no_gt'].get(c) is not None for c in JUDGE_CRITERIA)
]
parse_fail_rate = 1.0 - len(fully_parsed) / max(len(judge_ok), 1)
if parse_fail_rate > 0.30:
    verify_warnings.append(
        f'High judge parse-failure rate: {parse_fail_rate*100:.1f}% '
        '(>30%). Consider reviewing the judge prompt.'
    )

# Suspiciously perfect scores
for c in JUDGE_CRITERIA:
    c_vals = [
        s.get('judge_no_gt', {}).get(c)
        for s in judge_ok
        if (s.get('judge_no_gt') or {}).get(c) is not None
    ]
    if len(c_vals) > 3 and all(v == CFG.judge_score_max for v in c_vals):
        verify_warnings.append(
            f'All {len(c_vals)} samples scored {CFG.judge_score_max}/5 on {c} — '
            'check for prompt injection or judge collapse.'
        )

# Collect metric ranges
clip_vals  = [s['clip_i'] for s in v_ok if s.get('clip_i') is not None]
lpips_vals = [s['lpips']  for s in v_ok if s.get('lpips')  is not None]
ssim_vals  = [s['ssim']   for s in v_ok if s.get('ssim')   is not None]

print('Verification complete.')
print(f'  Errors   : {len(verify_errors)}')
print(f'  Warnings : {len(verify_warnings)}')
print(f'  Judge attempted   : {len(judge_ok)}')
print(f'  Fully parsed      : {len(fully_parsed)}')
print(f'  Parse-fail rate   : {parse_fail_rate*100:.1f}%')
if clip_vals:
    print(f'  CLIP-I range      : [{min(clip_vals):.3f}, {max(clip_vals):.3f}]')
if lpips_vals:
    print(f'  LPIPS  range      : [{min(lpips_vals):.3f}, {max(lpips_vals):.3f}]')
if ssim_vals:
    print(f'  SSIM   range      : [{min(ssim_vals):.3f}, {max(ssim_vals):.3f}]')
if verify_warnings:
    print('\nWarnings:')
    for w in verify_warnings:
        print(f'  WARN: {w}')
if verify_errors:
    print('\nErrors:')
    for e in verify_errors[:20]:
        print(f'  ERROR: {e}')
    raise RuntimeError(
        f'Verification failed with {len(verify_errors)} error(s).'
    )
print('\nAll verification checks passed. Benchmark evaluation complete.')
print(f'Final report: {CFG.outputs_bench / "bench_report.json"}')
print(f'Split        : {CFG.bench_split}')
print('Phase 4 v1.1 evaluation complete.')
